# CAPM LAB · IBEX 35
### Evaluación empírica del riesgo sistemático, el coste de capital histórico y las rentabilidades realizadas

**Proyecto de investigación cuantitativa en Python** · **Universo:** seis empresas en la demostración y componentes del IBEX 35 obtenidos dinámicamente para el análisis ampliado · **Mercado:** IBEX 35 Total Return, índice con dividendos reinvertidos · **Moneda:** EUR · **Frecuencia:** mensual.

Este estudio aplica el **Capital Asset Pricing Model (CAPM)** a empresas del IBEX 35 mediante tres fuentes diferenciadas: **(1) el CSV histórico del IBEX 35 Total Return exportado de Investing.com** —incluido en este repositorio— para las rentabilidades del mercado, **(2) Yahoo Finance** para las cotizaciones ajustadas de las acciones y **(3) el Banco Central Europeo (BCE)** para la serie de rendimiento de la deuda soberana española a diez años usada como aproximación de la tasa libre de riesgo. La composición del universo ampliado y sus sectores se obtienen mediante un extractor de **MarketScreener**. **FRED no se utiliza en la configuración predeterminada**: solo permanece disponible como fuente alternativa opcional de tipos de interés.

> **Pregunta central:** ¿Hasta qué punto el riesgo sistemático estimado por el CAPM explica las rentabilidades históricas de las empresas del IBEX 35? ¿Qué estabilidad y capacidad predictiva muestra el modelo al estimarse con ventanas móviles?

### Estructura del estudio

| Bloque | Contenido |
|---|---|
| 01–04 | Metodología, configuración, lectura del CSV, descargas y controles de calidad |
| 05 | CAPM por empresa: regresiones sobre excesos de retorno (OLS con covarianza HAC) |
| 06–07 | Demostración de seis empresas: SCL y Security Market Line (SML) |
| 08 | Estimaciones móviles de 36 meses y evaluación fuera de muestra |
| 09–13 | Universo ampliado extraído de MarketScreener: regresiones, SML, sectores y exportaciones |

### Ejecución reproducible

1. Clona o descarga este repositorio, instala las dependencias con `python -m pip install -r requirements.txt` y abre `notebooks/CAPM_IBEX35_CSV_TR_BCE.ipynb` desde la raíz del proyecto (consulta el [README](../README.md)).
2. El archivo **`data/ibex35_total_return_investing.csv` ya está incluido**. El notebook lo encuentra automáticamente a partir de rutas relativas tanto si se abre desde la raíz como desde `notebooks/`; también admite cargarlo en Google Colab.
3. Ejecuta las celdas en orden. Necesitarás conexión a internet para Yahoo Finance, BCE y MarketScreener. Los análisis dinámicos dependen de la disponibilidad de esos proveedores; no se rellenan los datos con observaciones inventadas. Las tablas, gráficos y Excel se guardan en `resultados_capm_demo/`.

> **Alcance:** estudio histórico e ilustrativo; no constituye una recomendación de inversión ni una valoración fundamental. La comparación entre retorno realizado y CAPM ajustado en la misma muestra es descriptiva y está estrechamente relacionada con el alfa OLS. El escenario de rf actual **no** es una estimación ex-ante independiente.

---

## 01 / Diseño del ejercicio

**Mercado y frecuencia.** Se utiliza el **IBEX 35 Total Return**, no el índice de precio `^IBEX`. Los 3.102 cierres diarios del CSV aportado se limpian respetando el formato numérico español y se convierten en cierres mensuales. Los precios de Yahoo Finance se descargan con `auto_adjust=True`. Se excluye el mes en curso para evitar observaciones incompletas.

**Tasa de referencia.** La configuración principal descarga la serie mensual del BCE `IRS/M.ES.L.L40.CI.0000.EUR.N.Z` (rendimiento de la deuda española a diez años). Se emplea un desfase de un mes respecto al retorno objetivo, evitando utilizar una observación de tipos publicada posteriormente. Para el CAPM mensual se transforma la tasa anual en tasa efectiva mensual antes de calcular excesos de retorno. **Precisión:** el rendimiento de un bono a diez años no equivale al rendimiento libre de riesgo de una inversión a un mes; incorpora duración y riesgo soberano. La elección es una aproximación documentada, no una identidad financiera.

**CAPM histórico.** Se estima `r_i,t − rf_t = α_i + β_i (r_m,t − rf_t) + ε_i,t` con OLS y errores estándar HAC/Newey–West (3 rezagos). Para comparar magnitudes se anualizan aritméticamente las medias mensuales (×12). El CAGR es geométrico y se presenta por separado. La prima histórica se estima a partir del **mercado y la rf alineados**, no mediante un porcentaje externo predeterminado.

**Sensibilidad a rf actual.** La comparación `Ke_hist` frente a `Ke_rf_actual` mantiene beta y prima históricas de cada empresa y reemplaza solamente la rf media por la última rf disponible del BCE. La diferencia, en puntos porcentuales, mide ese cambio mecánico; **no compara dos CAPM independientes ni predice retornos futuros**.

**Rolling.** Para el objetivo mensual `t`, la beta y prima se estiman usando los 36 meses anteriores; el contraste compara la predicción con el retorno posterior observado. Se exige historial mensual completo, sin rellenar huecos silenciosamente.

### Limitaciones e interpretación

- Los constituyentes actuales extraídos de MarketScreener no representan necesariamente la composición histórica del IBEX 35: existe **sesgo de supervivencia** si se extrapolan los resultados al pasado.
- Yahoo Finance y el CSV de Investing pueden aplicar convenciones distintas de ajustes o dividendos; se deben documentar y revisar discrepancias.
- La prima histórica no representa necesariamente una prima esperada futura. El ajuste dentro de muestra no demuestra capacidad predictiva.
- La rf basada en deuda española a diez años es una aproximación al coste de capital, no una tasa monetaria estrictamente libre de riesgo a un mes.
- El scraper de MarketScreener depende de una interfaz externa que puede cambiar, limitar consultas o no estar disponible.

### Fuentes

- [Investing.com · IBEX 35 Total Return (histórico)](https://es.investing.com/indices/ibex-total-return-historical-data)
- [BCE · tipos a largo plazo (IRS)](https://data.ecb.europa.eu/data/datasets/IRS)
- [Yahoo Finance](https://finance.yahoo.com/)
- [MarketScreener](https://es.marketscreener.com/)

**Alternativas opcionales, no utilizadas por defecto:** FRED/OCDE `IRLTLT01ESM156N` (España a diez años) y `IRSTCI01EZM156N` (interbancario del área euro). Las opciones se mantienen para pruebas de sensibilidad y requieren seleccionar expresamente `RF_PRESET`.


---

## 02 / Configuración y obtención de datos

Ejecuta el notebook desde el repositorio. Se instala el entorno **una sola vez** con `python -m pip install -r requirements.txt`, antes de arrancar Jupyter o Colab. El notebook utiliza el CSV incluido en `data/` y descarga en línea exclusivamente los datos que no forman parte del repositorio. En Colab, si se suben los archivos por separado, se admite la ruta `/content/Datos históricos del IBEX 35 con Div..csv` como alternativa.


In [ ]:
# Instala las dependencias fuera del notebook: python -m pip install -r requirements.txt
# En Colab, si fuera necesario, ejecuta antes en una celda separada:
# %pip install -q -r requirements.txt

# ==============================================================================
# 02 · CONFIGURACIÓN: IMPORTS, ESTILO Y PARÁMETROS
# ==============================================================================
import io
import warnings
from math import ceil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import statsmodels.api as sm
import yfinance as yf
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator, PercentFormatter

# Silencia el ruido de FutureWarning de pandas/yfinance (no afecta a los cálculos).
warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------------------------------------------- ESTILO
plt.rcParams.update({
    "figure.figsize": (12, 5.6), "figure.dpi": 115,
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.titlesize": 15, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": .17,
    "savefig.bbox": "tight", "savefig.facecolor": "white",
})
INK, BLUE, TEAL, GOLD, GREY = "#15243B", "#3765AD", "#169B91", "#D79431", "#6B788A"

# Directorios de salida
OUT_DIR = Path.cwd() / "resultados_capm_demo"
OUT_DIR.mkdir(parents=True, exist_ok=True)
# ---------------------------------------------------------------- PERIODO Y MODELO
BENCHMARK = "IBEX35_TR"
BENCHMARK_FUENTE = "Investing.com · IBEX 35 Total Return · IBEXTR"
BENCHMARK_URL = "https://es.investing.com/indices/ibex-total-return-historical-data"
# Ruta relativa portable para GitHub / Jupyter / VS Code y alternativa Colab.
CSV_CANDIDATES = [
    Path.cwd() / "data" / "ibex35_total_return_investing.csv",
    Path.cwd().parent / "data" / "ibex35_total_return_investing.csv",
    Path.cwd() / "ibex35_total_return_investing.csv",
    Path("/content/Datos históricos del IBEX 35 con Div..csv"),
]
CSV_INVESTING = next((p for p in CSV_CANDIDATES if p.is_file()), CSV_CANDIDATES[0])
START = "2015-01-01"        # primera rentabilidad de 2015
DOWNLOAD_START = "2014-12-01" # cierre base para enero de 2015
END = None                  # None = última fecha disponible
ANNUAL = 12                 # meses por año (anualización aritmética: 12 × media mensual)
WINDOW = 36                 # meses de la ventana del CAPM móvil
HAC_LAGS = 3                # rezagos Newey-West (errores robustos a autocorrelación)
MIN_OBS = 36                # universo: meses mínimos para estimar; si no, se excluye
MIN_COVERAGE = 0.95         # demo: cobertura mínima de retornos; si no, se detiene

# ---------------------------------------------------------------- TASA LIBRE DE RIESGO
# Ver la sección 01 para la justificación. Opciones: ES_10Y_ECB | ES_10Y_FRED | EA_OVERNIGHT_FRED
RF_PRESET = "ES_10Y_ECB"
RF_LAG_MESES = 1            # la rf del mes t es la publicada en t-1 (0 = anticipación, no usar)

RF_PRESETS = {
    "ES_10Y_ECB": dict(
        fuente="ecb", serie="IRS/M.ES.L.L40.CI.0000.EUR.N.Z",
        etiqueta="Bono español 10 años (BCE, rendimiento de convergencia, media mensual)"),
    "ES_10Y_FRED": dict(
        fuente="fred", serie="IRLTLT01ESM156N",
        etiqueta="Bono español 10 años (OCDE vía FRED, media mensual)"),
    "EA_OVERNIGHT_FRED": dict(
        fuente="fred", serie="IRSTCI01EZM156N",
        etiqueta="Interbancario overnight área euro (OCDE vía FRED) · serie original, no española"),
}

TICKERS = {
    "SAN.MC": "Banco Santander", "TEF.MC": "Telefónica",
    "ITX.MC": "Inditex", "REP.MC": "Repsol", "IBE.MC": "Iberdrola",
    "BBVA.MC": "BBVA",
}

DEMO_UNIVERSE = list(TICKERS)
def corto(t):
    # 'ITX.MC' -> 'ITX'
    return str(t).split(".")[0].upper()
assert len(DEMO_UNIVERSE) == 6, "La muestra de demostración debe contener 6 empresas."
assert len(set(DEMO_UNIVERSE)) == len(DEMO_UNIVERSE), "Tickers duplicados en el diccionario."

def nombre(ticker):
    """Devuelve el nombre corto de la empresa."""
    return TICKERS.get(ticker, ticker)

def corto(ticker):
    """Devuelve el símbolo sin el sufijo de mercado."""
    return ticker.removesuffix(".MC")
# ---------------------------------------------------------------- PRIMA DE MERCADO HISTÓRICA (se calcula después de alinear BCE y mercado)
# Sin prima ex-ante externa: se calcula directamente con los retornos del CSV y la rf BCE.

print("="*93)
print(f"CAPM LAB | {len(TICKERS)} compañías | rf={RF_PRESET} | ventana={WINDOW}m | destino={OUT_DIR}")
some_symbols = [*TICKERS, BENCHMARK]
# Solo los valores se descargan de Yahoo; IBEX35_TR procede del CSV de Investing.
raw = yf.download(list(TICKERS), start=DOWNLOAD_START, end=END, auto_adjust=True,
                  progress=False, threads=False, group_by="column")
if raw.empty:
    raise RuntimeError("Yahoo Finance no ha proporcionado precios de las empresas.")
if isinstance(raw.columns, pd.MultiIndex):
    if "Close" not in raw.columns.get_level_values(0):
        raise RuntimeError("Yahoo Finance no devolvió Close.")
    daily = raw["Close"].copy()
else:
    if len(TICKERS) != 1:
        raise RuntimeError("Formato inesperado de Yahoo Finance.")
    daily = raw[["Close"]].rename(columns={"Close": next(iter(TICKERS))})
missing = [t for t in TICKERS if t not in daily or daily[t].dropna().empty]
if missing:
    raise RuntimeError(f"Sin cotizaciones de: {missing}")
daily.index = pd.to_datetime(daily.index).tz_localize(None).normalize()
daily = daily.sort_index()[list(TICKERS)].apply(pd.to_numeric, errors="coerce")

# CSV de Investing: Fecha DD.MM.AAAA, Último en formato numérico español.
if not CSV_INVESTING.exists():
    raise FileNotFoundError(f"No se encuentra el CSV del benchmark: {CSV_INVESTING}")
tr_df = pd.read_csv(CSV_INVESTING, encoding="utf-8-sig", sep=None, engine="python")
tr_df.columns = (tr_df.columns.astype(str)
                 .str.replace("\ufeff", "", regex=False)
                 .str.replace('"', "", regex=False).str.strip())
if not {"Fecha", "Último"}.issubset(tr_df.columns):
    raise ValueError(f"Columnas CSV inesperadas: {list(tr_df.columns)}")
tr_fecha = pd.to_datetime(tr_df["Fecha"].astype(str).str.strip(),
                         format="%d.%m.%Y", errors="coerce")
tr_precio = pd.to_numeric(tr_df["Último"].astype(str)
    .str.replace("\u00a0", "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False), errors="coerce")
tr = pd.Series(tr_precio.to_numpy(), index=tr_fecha, name=BENCHMARK)
tr = tr.loc[tr.index.notna()].dropna().sort_index()
tr = tr[~tr.index.duplicated(keep="last")]
if len(tr) < 2:
    raise RuntimeError("El CSV Investing no contiene un histórico suficiente.")
# Unir cotizaciones reales, sin inventar ni rellenar cierres.
daily = daily.join(tr, how="outer").sort_index()
no_quote = [t for t in some_symbols if daily[t].dropna().empty]
if no_quote:
    raise RuntimeError(f"Activos sin cotizaciones: {no_quote}")
print(f"Benchmark IBEX TR: {len(tr)} cierres | {tr.index.min():%Y-%m-%d} - {tr.index.max():%Y-%m-%d}")

prices = daily.resample("ME").last()

# No incluir mes en curso: usar solo meses completos en el calendario local.
today = pd.Timestamp.today().normalize()
prices = prices.loc[prices.index < today.to_period("M").to_timestamp("M")]
returns = prices.pct_change(fill_method=None)
returns = returns.loc[returns.index >= pd.Timestamp(START)]
if returns.dropna(how="all").empty:
    raise RuntimeError("No hay retornos mensuales válidos.")
print("Cotizaciones:", prices.index.min().date(), "→", prices.index.max().date())
print("Precio y retornos disponibles por activo:")
print("="*93)
display(pd.DataFrame({"Precios":prices.notna().sum(), "Retornos":returns.notna().sum()}))

# ==============================================================================
# 04.2 · TASA LIBRE DE RIESGO (ESPAÑA) Y ALINEACIÓN SIN ANTICIPACIÓN
# ==============================================================================
UA = {"User-Agent": "Mozilla/5.0 (CAPM-LAB notebook)"}

def descargar_rf(preset):
    # Devuelve un pd.Series con el rendimiento ANUAL en decimales (0.032 = 3,2 %),
    # indexado por pd.Period mensual. Falla de forma explícita: no hay sustitutos silenciosos.
    cfg = RF_PRESETS[preset]
    try:
        if cfg["fuente"] == "ecb":
            flow, key = cfg["serie"].split("/", 1)
            r = requests.get(f"https://data-api.ecb.europa.eu/service/data/{flow}/{key}",
                             params={"format": "csvdata", "startPeriod": DOWNLOAD_START[:7]},
                             headers=UA, timeout=30)
            r.raise_for_status()
            df = pd.read_csv(io.StringIO(r.text))
            s = pd.Series(pd.to_numeric(df["OBS_VALUE"], errors="coerce").to_numpy(),
                          index=pd.PeriodIndex(df["TIME_PERIOD"], freq="M"))
        else:  # FRED, CSV público sin clave de API: 1ª columna = fecha, 2ª = valor
            r = requests.get("https://fred.stlouisfed.org/graph/fredgraph.csv",
                             params={"id": cfg["serie"]}, headers=UA, timeout=30)
            r.raise_for_status()
            df = pd.read_csv(io.StringIO(r.text), na_values=["."])
            s = pd.Series(pd.to_numeric(df.iloc[:, 1], errors="coerce").to_numpy(),
                          index=pd.DatetimeIndex(pd.to_datetime(df.iloc[:, 0])).to_period("M"))
    except Exception as exc:
        otros = [p for p in RF_PRESETS if p != preset]
        raise RuntimeError(
            f"No se pudo obtener la tasa libre de riesgo '{preset}' ({cfg['serie']}): {exc!r}. "
            f"Revisa la conexión o elige otra en RF_PRESET ({', '.join(otros)}); "
            "no se usan sustitutos silenciosos.") from exc
    s = s.dropna().groupby(level=0).last().sort_index()
    s = s[s.index >= pd.Period(DOWNLOAD_START[:7], freq="M")] / 100.0
    if s.empty:
        raise RuntimeError(f"La serie {cfg['serie']} no contiene datos desde {START}.")
    return s

rf_annual_month = descargar_rf(RF_PRESET)

periods = returns.index.to_period("M")
rf_calendar = rf_annual_month.reindex(pd.period_range(periods.min() - RF_LAG_MESES, periods.max(), freq="M"))
rf_annual_used = rf_calendar.shift(RF_LAG_MESES).reindex(periods)   # rf conocida al empezar el mes t
rf_annual_used.index = returns.index
rf_monthly = (1 + rf_annual_used).pow(1 / ANNUAL) - 1
if rf_monthly.notna().sum() < WINDOW + 12:
    raise RuntimeError("Cobertura insuficiente de la tasa libre de riesgo para el CAPM móvil.")

RF_ANUAL_ACTUAL = float(rf_annual_month.iloc[-1])        # última observación publicada
RF_FECHA_ACTUAL = rf_annual_month.index[-1]

# Intersección de mercado y rf; el soporte por activo se controla en cada estimación.
common = returns.join(rf_monthly.rename("rf_m")).dropna(subset=[BENCHMARK, "rf_m"])
PRIMA_HIST_GLOBAL = float((common[BENCHMARK] - common["rf_m"]).mean() * ANNUAL)
if len(common) < WINDOW + 12:
    raise RuntimeError(f"Histórico compartido insuficiente para ventana móvil de {WINDOW} meses.")
coverage = common[some_symbols].notna().mean().rename("Cobertura")
print("="*93)
print(f"Rf: {RF_PRESETS[RF_PRESET]['etiqueta']} | última observación fuente:", rf_annual_month.dropna().index[-1])
print("Último mes con tasa alineada:", rf_monthly.dropna().index[-1].strftime("%Y-%m"))
print(f"Meses elegibles: {len(common)} ({common.index.min():%Y-%m} / {common.index.max():%Y-%m})")
print("="*93)
display(coverage.to_frame().style.format("{:.1%}"))
insufficient = [s for s in TICKERS if coverage[s] < MIN_COVERAGE]
if insufficient:
    raise RuntimeError(f"Cobertura inferior a {MIN_COVERAGE:.0%}: {insufficient}")
print("="*93)
sin_rf = returns.index.difference(common.index)
print(f"Rf: {RF_PRESETS[RF_PRESET]['etiqueta']}")
print(f"Serie: {RF_PRESETS[RF_PRESET]['serie']} | última observación publicada: {RF_FECHA_ACTUAL} ({RF_ANUAL_ACTUAL:.2%})")
print(f"Rf anual media de la muestra: {rf_annual_used.dropna().mean():.2%} "
      f"(mín {rf_annual_used.min():.2%} · máx {rf_annual_used.max():.2%})")
if len(sin_rf):
    print(f"Meses excluidos por falta de rf: {len(sin_rf)} ({sin_rf.min():%Y-%m} → {sin_rf.max():%Y-%m})")
print("="*93)
if rf_monthly.notna().sum() < WINDOW + 12:
    raise RuntimeError("Cobertura insuficiente de la tasa de referencia para el CAPM móvil.")


def rolling_capm(frame, ticker, window=36, hac_lags=3):
    """Estimaciones ex ante usando W meses inmediatamente anteriores al mes objetivo."""
    records = []
    required = [ticker, BENCHMARK, "rf_m"]
    for j in range(window, len(frame)):
        hist = frame.iloc[j-window:j][required]
        target = frame.iloc[j]
        # Ventanas calendario completas: no se eliminan huecos silenciosamente.
        if hist.isna().any().any() or target[required].isna().any():
            continue
        x = (hist[BENCHMARK] - hist["rf_m"]).to_numpy(dtype=float)
        y = (hist[ticker] - hist["rf_m"]).to_numpy(dtype=float)
        if np.var(x) < 1e-12:
            continue
        fit = sm.OLS(y, sm.add_constant(x)).fit(cov_type="HAC", cov_kwds={"maxlags": min(hac_lags, window-2)})
        alpha, beta = map(float, fit.params)
        premium = float(x.mean())
        predicted = float(target["rf_m"] + beta * premium)
        realized = float(target[ticker])
        ci = fit.conf_int(alpha=.05)
        records.append({
            "Fecha": frame.index[j], "Ticker": ticker, "Empresa": TICKERS[ticker],
            "Ventana": window, "Alpha_m": alpha, "Beta_rolling": beta,
            "R2_rolling": float(fit.rsquared), "Beta_se_HAC": float(fit.bse[1]),
            "Beta_p_HAC": float(fit.pvalues[1]),
            "Beta_IC95_inf": float(ci[1,0]), "Beta_IC95_sup": float(ci[1,1]),
            "Prima_mensual_prev": premium, "Rf_mensual": float(target["rf_m"]),
            "CAPM_m": predicted, "Real_m": realized, "Diferencia_m": realized-predicted,
        })
    return pd.DataFrame.from_records(records)

# Reindexación explícita para que una ventana tenga exactamente W meses naturales consecutivos.
full_calendar = pd.date_range(common.index.min(), common.index.max(), freq="ME")
base = common.reindex(full_calendar)
paths = pd.concat([rolling_capm(base, t, WINDOW) for t in TICKERS], ignore_index=True)
if paths.empty:
    raise RuntimeError("Sin estimaciones rolling: revisa cobertura y fechas.")
paths = paths.sort_values(["Ticker", "Fecha"]).reset_index(drop=True)
display(paths.groupby(["Ticker","Empresa"]).agg(Primer_mes=("Fecha","min"), Ultimo_mes=("Fecha","max"), Observaciones=("Fecha","size")))
print("="*93)

## 05 / Estimación del CAPM para la muestra de demostración

**Regresión sobre excesos de retorno mensuales:**

$$r_{i,t}-r_{f,t}=\alpha_i+\beta_i(r_{m,t}-r_{f,t})+\varepsilon_{i,t}$$

**CAPM histórico (muestra válida individual):**

$$K_{e,i}^{hist}=\overline{r_f}^{\,anual}+\hat\beta_i\,\overline{(r_m-r_f)}^{\,anual}$$

**Sensibilidad sustituyendo únicamente la rf media por la última tasa disponible del BCE:**

$$K_{e,i}^{rf\ actual}=r_{f,actual}+\hat\beta_i\,\overline{(r_m-r_f)}^{\,anual}$$

Se mantiene la misma beta y prima histórica. La diferencia entre ambas estimaciones **no** es una validación independiente del CAPM ni un escenario prospectivo completo. La estimación usa OLS con errores HAC/Newey–West. La Parte I comienza con seis empresas; el universo completo se extrae y estima en la Parte II. Las empresas con menos de `MIN_OBS` meses válidos se excluyen y documentan.


In [ ]:
# ==============================================================================
# 05.1 · ESTIMACIÓN CAPM POR EMPRESA (OLS + HAC)
# ==============================================================================
RF_MEDIA_GLOBAL = float(common["rf_m"].mean() * ANNUAL)
MKT_MEDIA_GLOBAL = float(common[BENCHMARK].mean() * ANNUAL)
PRIMA_HIST_GLOBAL = MKT_MEDIA_GLOBAL - RF_MEDIA_GLOBAL

def fit_capm(frame, ticker):
    """
    Estima el CAPM mediante OLS con errores robustos HAC.

    Devuelve:
        row : diccionario de indicadores
        x   : exceso de retorno del mercado
        y   : exceso de retorno de la empresa
        fit : resultado completo de la regresión
    """
    d = frame[[ticker, BENCHMARK, "rf_m"]].dropna()
    if len(d) < MIN_OBS:
        return None
    y = (d[ticker] - d["rf_m"]).astype(float)
    x = (d[BENCHMARK] - d["rf_m"]).astype(float)
    if x.var() < 1e-12:
        return None
    model = sm.OLS(
        y,
        sm.add_constant(x)
    ).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": HAC_LAGS}
    )

    alpha = float(model.params.iloc[0])
    beta = float(model.params.iloc[1])
    ci = model.conf_int(alpha=0.05)
    # Rentabilidades y riesgo anualizados
    rf_a = float(d["rf_m"].mean() * ANNUAL)
    m_a = float(d[BENCHMARK].mean() * ANNUAL)
    real_a = float(d[ticker].mean() * ANNUAL)
    prima = m_a - rf_a
    capm_a = rf_a + beta * prima
    capm_rf_actual = RF_ANUAL_ACTUAL + beta * prima
    vol = float(d[ticker].std() * np.sqrt(ANNUAL))
    sharpe = (real_a - rf_a) / vol if vol > 0 else np.nan
    cagr = float(
        (1 + d[ticker]).prod() ** (ANNUAL / len(d)) - 1
    )
    beta_cov = float(
        d[ticker].cov(d[BENCHMARK]) / d[BENCHMARK].var()
    )
    row = {
        # Identificación
        "Ticker": ticker,
        "Empresa": TICKERS[ticker],
        "N_meses": len(d),
        # Regresión
        "Beta": beta,
        "Alfa_anual": alpha * ANNUAL,
        "P_alfa_HAC": float(model.pvalues.iloc[0]),
        "P_beta_HAC": float(model.pvalues.iloc[1]),
        "SE_beta_HAC": float(model.bse.iloc[1]),
        "T_beta_HAC": float(model.tvalues.iloc[1]),
        # Intervalos de confianza
        "Beta_IC95_inf": float(ci.iloc[1, 0]),
        "Beta_IC95_sup": float(ci.iloc[1, 1]),
        "Alfa_IC95_inf_m": float(ci.iloc[0, 0]),
        "Alfa_IC95_sup_m": float(ci.iloc[0, 1]),
        # Calidad y controles
        "R2": float(model.rsquared),
        "Beta_cov": beta_cov,
        "Volatilidad": vol,
        "Sharpe": sharpe,
        # CAPM histórico
        "Rf_anual": rf_a,
        "Mercado_anual": m_a,
        "Prima_anual": prima,
        "CAPM_anual": capm_a,
        # Rentabilidad realizada
        "Real_anual_arit": real_a,
        "CAGR_real": cagr,
        "Gap_pp": 100 * (real_a - capm_a),
        # Alias para las nuevas visualizaciones
        "Ke_hist": capm_a,
        "Ret_real_arit": real_a,
        "Rf_media_hist": rf_a,
        "Prima_hist": prima,
        "Ke_rf_actual": capm_rf_actual,
        "Delta_Ke_pp": 100 * (capm_rf_actual - capm_a),
    }
    return row, x, y, model

# ==============================================================================
# ESTIMACIÓN Y ALMACENAMIENTO
# ==============================================================================
rows = []
store = {}
excluded = []
for ticker in TICKERS:
    out = fit_capm(common, ticker)
    if out is None:
        excluded.append({
            "Ticker": ticker,
            "Empresa": TICKERS[ticker],
            "Meses_validos": int(
                common[ticker].notna().sum()
            ),
            "Motivo": "Observaciones insuficientes o mercado sin varianza"
        })
        continue

    row, x, y, model = out
    rows.append(row)
    store[ticker] = (x, y, model)
if not rows:
    raise RuntimeError("No se ha podido estimar ninguna regresión CAPM.")
summary = pd.DataFrame(rows)
res = summary.copy()
excluded_df = pd.DataFrame(excluded)

res_demo = res.set_index("Ticker").loc[DEMO_UNIVERSE].reset_index()
# ==============================================================================
# PRESENTACIÓN DE RESULTADOS
# ==============================================================================
show_cols = [
    "Ticker", "Empresa", "N_meses", "Beta", "R2",
    "Rf_anual", "Prima_anual", "CAPM_anual",
    "Real_anual_arit", "CAGR_real", "Gap_pp"
]

print("=" * 120)
print(
    f"Periodo: {common.index.min():%Y-%m} "
    f"— {common.index.max():%Y-%m}"
)
print(
    f"Empresas estimadas: {len(summary)} | "
    f"Excluidas: {len(excluded_df)}"
)
print("=" * 120)
if not excluded_df.empty:
    display(excluded_df)

print("ESTIMACIÓN CAPM · IBEX 35")
print("=" * 120)
print(f"Periodo común            : ({len(common)} meses)  |   {common.index.max():%Y-%m} {common.index.min():%Y-%m}                         |")
print(f"Rf media de la muestra   : {RF_MEDIA_GLOBAL:.2%}        |   Rf actual ({RF_FECHA_ACTUAL}): {RF_ANUAL_ACTUAL:.2%}              |")
print(f"Mercado (media anual)    : {MKT_MEDIA_GLOBAL:.2%}       |-------------------------------------------|")
print(f"Prima histórica IBEX TR  : {PRIMA_HIST_GLOBAL:.2%}        |                                           |")
print(f"Empresas estimadas       : {len(res)}            |   excluidas: {len(excluded_df)}                            |")

print("=" * 120)
print("PARÁMETROS CAPM · MUESTRA HISTÓRICA")
print("=" * 120)
print("-" * 120)
display(
    summary[show_cols].style.format({
        "Beta": "{:.2f}",
        "R2": "{:.2f}",
        "Rf_anual": "{:.2%}",
        "Prima_anual": "{:.2%}",
        "CAPM_anual": "{:.2%}",
        "Real_anual_arit": "{:.2%}",
        "CAGR_real": "{:.2%}",
        "Gap_pp": "{:+.2f}"
    })
)
print("-" * 120)

# ==============================================================================
# 05.2 · FUNCIONES DE VISUALIZACIÓN (se reutilizan en las Partes I y II)
# ==============================================================================
try:
    from adjustText import adjust_text
    HAS_ADJUST_TEXT = True
except ImportError:
    HAS_ADJUST_TEXT = False
    print("adjustText no está instalado: las etiquetas de la SML no se reajustarán (opcional).")


def draw_scl(ax, x, y, fit, titulo, compact=True):
    # Security Characteristic Line: dispersión de excesos de retorno y recta OLS.
    a, b = float(fit.params.iloc[0]), float(fit.params.iloc[1])
    grid = np.linspace(float(x.min()), float(x.max()), 100)
    ax.scatter(x, y, s=18 if compact else 30, color=BLUE, alpha=.55, edgecolors="white", linewidths=.3)
    ax.plot(grid, a + b * grid, color=TEAL, lw=2.3 if compact else 2.7)
    ax.axhline(0, color=GREY, lw=.8, ls="--", alpha=.6)
    ax.axvline(0, color=GREY, lw=.8, ls="--", alpha=.6)
    ax.set_title(titulo, loc="left", pad=8)
    ax.text(.03, .96,
            f"β = {b:.2f}   ·   α = {a * ANNUAL:+.1%}/año\n"
            f"R² = {fit.rsquared:.2f}   ·   p(β) HAC = {fit.pvalues.iloc[1]:.3g}   ·   n = {len(x)}",
            transform=ax.transAxes, ha="left", va="top", fontsize=8.5 if compact else 10, color=INK,
            bbox=dict(boxstyle="round,pad=.4", facecolor="white", edgecolor="#DCE4EC", alpha=.93))
    ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
    ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
    ax.set_xlabel(f"Exceso de retorno del mercado ({BENCHMARK}) · mensual")
    ax.set_ylabel("Exceso de retorno de la acción · mensual")


def panel_scl(tickers, titulo, ruta, ncols=2, alto=4.4):
    # Panel de regresiones CAPM (una celda por empresa) guardado en `ruta`.
    n = len(tickers)
    nrows = ceil(n / ncols)
    fig, axs = plt.subplots(nrows, ncols, figsize=(6.9 * ncols, alto * nrows),
                            constrained_layout=True, squeeze=False)
    for ax, t in zip(axs.flat, tickers):
        x, y, fit = store[t]
        draw_scl(ax, x, y, fit, f"{nombre(t)} ({corto(t)})")
    for ax in axs.flat[n:]:
        ax.axis("off")
    fig.suptitle(titulo, fontsize=17, weight="bold", color=INK)
    fig.savefig(ruta, dpi=150)
    plt.show()
    print(f"Guardado: {ruta}")


def plot_sml(df, titulo, ruta, figsize=(14, 8), etiquetar=True):
    # SML: CAPM histórico (círculos) frente al retorno aritmético realizado (rombos).
    # La línea turquesa es una SML de REFERENCIA con la rf y la prima de la muestra común;
    # cada empresa usa su propia muestra, por lo que las de menor historia pueden desviarse.
    d = df.dropna(subset=["Beta", "Ke_hist", "Ret_real_arit"]).sort_values("Beta").reset_index(drop=True)
    b = d["Beta"].to_numpy(float)
    ke = d["Ke_hist"].to_numpy(float) * 100
    rr = d["Ret_real_arit"].to_numpy(float) * 100

    fig, ax = plt.subplots(figsize=figsize)
    ax.vlines(b, np.minimum(ke, rr), np.maximum(ke, rr), colors=GREY, alpha=.42, lw=1.3, zorder=2)
    ax.scatter(b, ke, color=BLUE, s=75, zorder=4)
    ax.scatter(b, rr, color=GOLD, marker="D", s=65, zorder=5)

    x_min, x_max = min(0, b.min() - .12), max(1.8, b.max() + .12)
    xgrid = np.linspace(x_min, x_max, 250)
    sml = 100 * (RF_MEDIA_GLOBAL + xgrid * PRIMA_HIST_GLOBAL)
    ax.plot(xgrid, sml, color=TEAL, lw=2.5, zorder=3)

    y_all = np.concatenate([ke, rr, sml])
    y_rng = max(float(y_all.max() - y_all.min()), 1.0)
    ax.set_xlim(x_min - .03, x_max + .06)
    ax.set_ylim(min(0, float(y_all.min()) - .06 * y_rng), float(y_all.max()) + .10 * y_rng)
    ax.axhline(0, color=GREY, lw=.8, ls="--", alpha=.6)
    ax.axvline(0, color=GREY, lw=.8, ls="--", alpha=.6)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=9))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=8))

    texts = [ax.text(bi + .008 * (x_max - x_min), ri + .012 * y_rng, corto(t),
                     fontsize=8.5, color=INK, alpha=.92, ha="left", va="bottom", zorder=6)
             for bi, ri, t in zip(b, rr, d["Ticker"])]
    if etiquetar and HAS_ADJUST_TEXT:
        try:   # ajuste suave y solo vertical, sin flechas
            adjust_text(texts, ax=ax,
                        only_move={"text": "y", "static": "y", "explode": "y", "pull": "y"},
                        force_text=(0.0, 0.08), force_static=(0.0, 0.025),
                        force_pull=(0.0, 0.03), force_explode=(0.0, 0.02),
                        max_move=(0, 2), iter_lim=35, expand_axes=False, ensure_inside_axes=True)
        except TypeError:
            pass   # versión antigua de adjustText: se dejan las etiquetas sin reajustar

    ax.set_xlabel("Beta estimada · riesgo sistemático", fontsize=12)
    ax.set_ylabel("Retorno anual aritmético (%)", fontsize=12)
    ax.set_title(titulo, fontsize=15, pad=16, color=INK)
    ax.legend(handles=[
        Line2D([], [], marker="o", ls="None", markerfacecolor=BLUE, markeredgecolor=BLUE, markersize=9,
               label="CAPM histórico (exigido)"),
        Line2D([], [], marker="D", ls="None", markerfacecolor=GOLD, markeredgecolor=GOLD, markersize=8,
               label="Retorno real (aritmético)"),
        Line2D([], [], color=GREY, lw=1.5, label="Diferencia real − CAPM"),
        Line2D([], [], color=TEAL, lw=2.5, label="SML de referencia (muestra común)")],
        loc="upper left", fontsize=10, frameon=True, facecolor="white", edgecolor="#DCE4EC")
    fig.tight_layout()
    fig.savefig(ruta, dpi=170)
    plt.show()
    print(f"Guardado: {ruta}")

print("=" * 120)

print(
    "Ke histórico = Rf media histórica individual "
    "+ Beta × Prima histórica individual"
)
print(
    "Ke con Rf actual = Rf actual BCE "
    "+ Beta × Prima histórica individual"
)
print(
    "Δ Ke (pp) = Ke con Rf actual − Ke histórico"
)
print(
    "Gap (pp) = Retorno real aritmético − Ke histórico"
)
print("=" * 120)

In [ ]:
# Regresión lineal CAPM: seis dispersogramas, líneas OLS y resumen inferencial.
from matplotlib.ticker import PercentFormatter
print("="*130)
print("Regresiones CAPM-Sensibilidad al mercado.")
print("="*130)
fig, axs = plt.subplots(3, 2, figsize=(13.8, 14), constrained_layout=True)
for ax, (ticker, name) in zip(axs.flat, TICKERS.items()):
    d = common[[ticker, BENCHMARK, "rf_m"]].dropna()
    x = (d[BENCHMARK] - d["rf_m"]).astype(float)
    y = (d[ticker] - d["rf_m"]).astype(float)
    fit = sm.OLS(y, sm.add_constant(x)).fit(cov_type="HAC", cov_kwds={"maxlags": 3})
    grid = np.linspace(x.min(), x.max(), 100)
    line = float(fit.params.iloc[0]) + float(fit.params.iloc[1]) * grid
    ax.scatter(x, y, alpha=.50, color=BLUE, edgecolors="white", linewidths=.3, s=32)
    ax.plot(grid, line, color=TEAL, lw=2.5, label="Recta OLS")
    ax.axhline(0, color=GREY, lw=.7, alpha=.6)
    ax.axvline(0, color=GREY, lw=.7, alpha=.6)
    ax.set_title(name, loc="left", pad=11)
    ax.text(.03, .96, f"β = {fit.params.iloc[1]:.2f}   ·   α = {fit.params.iloc[0]:+.2%}/mes\n"
            f"R² = {fit.rsquared:.2f}   ·   p(β) HAC = {fit.pvalues.iloc[1]:.3g}   ·   n = {len(d)}",
            transform=ax.transAxes, ha="left", va="top", fontsize=9, color=INK,
            bbox=dict(boxstyle="round,pad=.4", facecolor="white", edgecolor="#DCE4EC", alpha=.93))
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.set_xlabel("Exceso de retorno IBEX 35 · mensual")
    ax.set_ylabel("Exceso de retorno acción · mensual")
for ax in axs.flat[len(TICKERS):]:
    ax.axis("off")
fig.suptitle("Regresiones CAPM | sensibilidad al mercado", fontsize=19, weight="bold", color=INK)
fig.savefig(OUT_DIR / "02_regresiones_capm.png", dpi=180)
plt.show()

reg_cols = ["Empresa", "N_meses", "Beta", "SE_beta_HAC", "T_beta_HAC", "P_beta_HAC",
            "Beta_IC95_inf", "Beta_IC95_sup", "Alfa_anual", "P_alfa_HAC", "R2"]
regression_table = summary[reg_cols].copy()
print("="*130)
display(regression_table.style.format({
    "Beta":"{:.3f}","SE_beta_HAC":"{:.3f}","T_beta_HAC":"{:.2f}",
    "P_beta_HAC":"{:.4g}", "Beta_IC95_inf":"{:.3f}", "Beta_IC95_sup":"{:.3f}",
    "Alfa_anual":"{:+.2%}","P_alfa_HAC":"{:.4g}","R2":"{:.3f}"}))
print("="*130)

In [ ]:
# ==============================================================================
# 07 · SECURITY MARKET LINE (DEMO)
# ==============================================================================
plot_sml(res_demo, "Security Market Line | Demostración (6 empresas)\nRetorno exigido por CAPM vs. retorno realizado",
         OUT_DIR / "02_sml_demo.png", figsize=(11, 6.5))

## 08 / CAPM móvil sin anticipación y diagnóstico

Es la prueba más exigente: para cada mes `t` se estima el CAPM **solo con los 36 meses anteriores** y se compara lo que habría exigido con lo que ocurrió después.

$$ \hat r_{i,t}=r_{f,t}+\hat\beta_{i,t}\cdot\overline{(r_m-r_f)}_{\,t-36:t-1} $$

**Diagnóstico** (por empresa, sobre todos los meses evaluados):

- **Dif. anual (pp):** `12 × media(real − exigido)`. Positiva = la acción rindió más de lo que el CAPM exigía.
- **p-valor (HAC):** contraste de que esa diferencia media es cero.
- **% meses real > CAPM:** frecuencia de superación.
- **R² fuera de muestra vs. rf:** `1 − Σ(real − CAPM)² / Σ(real − rf)²`. Positivo = el CAPM predice mejor que suponer que la acción solo rinde la `rf`; negativo, peor.

Se calcula para `ROLLING_TICKERS` (por defecto, la muestra de seis valores ya descargada). El universo ampliado se descarga posteriormente en una sección separada; para extender el rolling a ese universo habría que integrarlo expresamente con esa matriz de retornos.

In [ ]:
# ==============================================================================
# 08.1 · CAPM MÓVIL (ROLLING) SIN ANTICIPACIÓN
# ==============================================================================
# Selección de la muestra para esta sección
ROLLING_TICKERS = list(DEMO_UNIVERSE)

def rolling_capm(frame, ticker, window=WINDOW, hac_lags=HAC_LAGS):
    """
    Estima un CAPM rolling utilizando exclusivamente los `window`
    meses anteriores al mes objetivo.
    La beta y la prima de mercado se estiman con información
    histórica. El retorno objetivo no interviene en la estimación.
    """
    records = []
    required = [ticker, BENCHMARK, "rf_m"]
    for j in range(window, len(frame)):
        hist = frame.iloc[j-window:j][required]
        target = frame.iloc[j]

        # Se exige una ventana completa y un mes objetivo válido
        if hist.isna().any().any() or target[required].isna().any():
            continue
        x = (
            hist[BENCHMARK] - hist["rf_m"]
        ).to_numpy(dtype=float)

        y = (
            hist[ticker] - hist["rf_m"]
        ).to_numpy(dtype=float)
        if np.var(x) < 1e-12:
            continue
        fit = sm.OLS(
            y,
            sm.add_constant(x)
        ).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": min(hac_lags, window - 2)}
        )
        alpha, beta = map(float, fit.params)
        # Prima mensual estimada con los meses anteriores
        premium = float(x.mean())
        # Predicción del CAPM para el mes objetivo
        predicted = float(
            target["rf_m"] + beta * premium
        )

        realized = float(target[ticker])
        ci = fit.conf_int(alpha=0.05)

        records.append({
            "Fecha": frame.index[j],
            "Ticker": ticker,
            "Empresa": TICKERS[ticker],
            "Ventana": window,

            "Alpha_m": alpha,
            "Beta_rolling": beta,
            "R2_rolling": float(fit.rsquared),

            "Beta_se_HAC": float(fit.bse[1]),
            "Beta_p_HAC": float(fit.pvalues[1]),
            "Beta_IC95_inf": float(ci[1, 0]),
            "Beta_IC95_sup": float(ci[1, 1]),

            "Prima_mensual_prev": premium,
            "Rf_mensual": float(target["rf_m"]),

            "CAPM_m": predicted,
            "Real_m": realized,
            "Diferencia_m": realized - predicted,
        })

    return pd.DataFrame.from_records(records)

# ==============================================================================
# CONSTRUCCIÓN DE LAS SERIES ROLLING
# ==============================================================================
full_calendar = pd.date_range(
    common.index.min(),
    common.index.max(),
    freq="ME"
)
base = common.reindex(full_calendar)
rolling_results = [
    rolling_capm(base, ticker)
    for ticker in ROLLING_TICKERS
]
rolling_results = [
    df for df in rolling_results if not df.empty
]
if not rolling_results:
    raise RuntimeError(
        "Sin estimaciones rolling: revisa cobertura y fechas."
    )
paths = pd.concat(
    rolling_results,
    ignore_index=True
)
orden_roll = {
    ticker: i
    for i, ticker in enumerate(ROLLING_TICKERS)
}
paths["_orden"] = paths["Ticker"].map(orden_roll)
paths = (
    paths.sort_values(["_orden", "Fecha"])
         .drop(columns="_orden")
         .reset_index(drop=True)
)

# ==============================================================================
# DIAGNÓSTICO DEL CAPM ROLLING
# ==============================================================================
def diagnostico_rolling(p):
    rows_d = []
    for ticker, g in p.groupby("Ticker", sort=False):
        if len(g) < 12:
            continue
        dif = g["Diferencia_m"].to_numpy(float)
        hac = sm.OLS(
            dif,
            np.ones(len(dif))
        ).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": HAC_LAGS}
        )
        sse = float(
            np.sum((g["Real_m"] - g["CAPM_m"]) ** 2)
        )

        sst = float(
            np.sum((g["Real_m"] - g["Rf_mensual"]) ** 2)
        )
        rows_d.append({
            "Ticker": ticker,
            "Empresa": TICKERS[ticker],

            "Primer_mes": g["Fecha"].min(),
            "Ultimo_mes": g["Fecha"].max(),
            "Meses": len(g),

            "Beta_media": g["Beta_rolling"].mean(),
            "Beta_min": g["Beta_rolling"].min(),
            "Beta_max": g["Beta_rolling"].max(),

            "Exigido_anual": g["CAPM_m"].mean() * ANNUAL,
            "Real_anual": g["Real_m"].mean() * ANNUAL,

            "Dif_anual_pp": 100 * dif.mean() * ANNUAL,
            "P_dif_HAC": float(hac.pvalues[0]),

            "Pct_real_sup": float((dif > 0).mean()),

            "R2_OOS_vs_rf": (
                1 - sse / sst if sst > 0 else np.nan
            ),
        })

    return pd.DataFrame(rows_d)
diag_rolling = diagnostico_rolling(paths)

# ==============================================================================
# PRESENTACIÓN DE RESULTADOS
# ==============================================================================
print("=" * 168)
print(
    f"CAPM MÓVIL · VENTANA {WINDOW} MESES · "
    f"{paths['Fecha'].min():%Y-%m} → "
    f"{paths['Fecha'].max():%Y-%m}"
)
print(
    f"Empresas analizadas: {paths['Ticker'].nunique()} "
    f"| Observaciones rolling: {len(paths)}"
)
print("=" * 168)
print("-" * 167)
if not diag_rolling.empty:
    display(
        diag_rolling.style.format({
            "Primer_mes": "{:%Y-%m}",
            "Ultimo_mes": "{:%Y-%m}",
            "Beta_media": "{:.2f}",
            "Beta_min": "{:.2f}",
            "Beta_max": "{:.2f}",
            "Exigido_anual": "{:.2%}",
            "Real_anual": "{:.2%}",
            "Dif_anual_pp": "{:+.2f}",
            "P_dif_HAC": "{:.3f}",
            "Pct_real_sup": "{:.0%}",
            "R2_OOS_vs_rf": "{:+.3f}"
        })
        .background_gradient(
            subset=["Dif_anual_pp"],
            cmap="RdYlGn"
        )
        .hide(axis="index")
    )
print("-" * 167)
print("=" * 168)

In [ ]:
"""
Extractor de datos del IBEX 35 desde Market Screener (API directa)
+ Enriquecimiento con tickers/betas de Yahoo Finance y betas sectoriales ponderadas.

Usa curl_cffi para imitar la huella TLS de un navegador real y evitar el 403.
Requiere: pip install -U curl_cffi pandas numpy openpyxl yfinance
"""
import json
import random
import re
import time
from datetime import datetime

import numpy as np
import pandas as pd
import yfinance as yf
from curl_cffi import requests as cffi_requests


class IBEXHeatmapExtractor:
    def __init__(self):
        self.url_base = "https://es.marketscreener.com"
        self.url_heatmap = f"{self.url_base}/cotizacion/indice/IBEX-35-7629/heatmap/"
        self.url_api = f"{self.url_base}/async/graph/heatmap/components-perf"
        self.params = {
            'codezb': '7629',     # IBEX 35
            'grouping': 'sbI',    # Agrupación por sector
            'varia': 'F271',      # Tipo de variación
            'weight': 'capi'      # Peso por capitalización
        }

        # Cabeceras mínimas: User-Agent, Accept-Encoding, Sec-Ch-Ua*, Sec-Fetch-*
        # los genera curl_cffi de forma coherente con el navegador impersonado.
        self.headers_html = {
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            'Accept-Language': 'es-ES,es;q=0.9,en;q=0.8',
        }
        self.headers_api = {
            'Accept': 'application/json, text/javascript, */*; q=0.01',
            'Accept-Language': 'es-ES,es;q=0.9,en;q=0.8',
            'Referer': self.url_heatmap,
            'X-Requested-With': 'XMLHttpRequest',
        }

        self.sectores = {}
        self.acciones = []
        self.raw_data = None
        self._cache_info = {}
        self.session = cffi_requests.Session(impersonate="chrome")

        # Mapeo mnemo (Market Screener) -> ticker Yahoo (.MC).
        self.yahoo_map_overrides = {
            "ITX": "ITX.MC", "SAN": "SAN.MC", "BBVA": "BBVA.MC", "CABK": "CABK.MC",
            "IBE": "IBE.MC", "REP": "REP.MC", "TEF": "TEF.MC", "FER": "FER.MC",
            "AENA": "AENA.MC", "ACS": "ACS.MC", "MAP": "MAP.MC", "BKT": "BKT.MC",
            "IAG": "IAG.MC", "CLNX": "CLNX.MC", "NTGY": "NTGY.MC", "ENG": "ENG.MC",
            "ELE": "ELE.MC", "ANA": "ANA.MC", "ANE": "ANE.MC", "ACX": "ACX.MC",
            "SOL": "SLR.MC", "SLR": "SLR.MC", "LOG": "LOG.MC", "ROVI": "ROVI.MC",
            "GRF": "GRF.MC", "IDR": "IDR.MC", "MRL": "MRL.MC", "COL": "COL.MC",
            "SCYR": "SCYR.MC", "UNI": "UNI.MC", "PUIG": "PUIG.MC",
            "MT": "MTS.MC", "MTS": "MTS.MC",
            "RED": "RED.MC", "SAB": "SAB.MC", "FDR": "FDR.MC", "AMS": "AMS.MC",
        }

    # ------------------------------------------------------------------ utilidades
    def _format_market_cap(self, value):
        """Capitalización en millones de euros con separador de miles español.
        (Se evita 'B€' porque en español 'billón' = 10^12 y confunde.)"""
        if pd.isna(value) or value == 0:
            return "0 M€"
        millones = value / 1_000_000
        if millones >= 100:
            return f"{millones:,.0f}".replace(",", ".") + " M€"
        return f"{millones:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".") + " M€"

    def _extraer_campos(self, item):
        """Extrae (ticker, nombre, link) tolerando cambios en el JSON de Market Screener.
        Ahora la API devuelve el ticker en 'name' y no trae 'mnemo' ni 'link'."""
        name = str(item.get('name') or '').strip()
        ticker = str(item.get('mnemo') or item.get('ticker') or item.get('symbol') or '').strip()

        nombre = ''
        for k in ('fullname', 'full_name', 'longName', 'title', 'label', 'company'):
            if item.get(k):
                nombre = str(item[k]).strip()
                break

        if not ticker:
            if re.fullmatch(r"[A-Z0-9\.]{1,6}", name):
                ticker = name           # 'name' contiene en realidad el ticker
            else:
                nombre = nombre or name
        elif not nombre and name != ticker:
            nombre = name

        link = ''
        for k in ('link', 'url', 'href'):
            if item.get(k):
                link = str(item[k]).strip()
                break
        if link.startswith('/'):
            link = f"{self.url_base}{link}"
        return ticker, nombre, link

    # ------------------------------------------------------- OBTENCIÓN Y PROCESADO
    def visitar_pagina_principal(self, session=None):
        session = session or self.session
        try:
            response = session.get(self.url_heatmap, headers=self.headers_html, timeout=15)
            if response.status_code == 200:
                print("Sesión establecida correctamente")
                time.sleep(random.uniform(1.5, 3.0))
                return True
            print(f"⚠️  Código de estado: {response.status_code}")
            return False
        except Exception as e:
            print(f"⚠️  Error al visitar página principal: {e}")
            return False

    def _pedir_api(self, session):
        response = session.get(self.url_api, params=self.params,
                               headers=self.headers_api, timeout=15)
        if response.status_code != 200:
            print(f"   ⚠️  Código: {response.status_code}")
            return None
        try:
            data = response.json()
        except Exception:
            print("   ⚠️  La respuesta no es JSON (posible página de bloqueo)")
            return None
        if data.get('error', True):
            print("   ⚠️  La API devolvió error")
            return None
        return data

    def obtener_datos(self):
        print("Obteniendo datos desde Market Screener API...")
        if not self.visitar_pagina_principal():
            print("⚠️  Continuando de todas formas...")
        try:
            time.sleep(random.uniform(1.0, 2.0))
            data = self._pedir_api(self.session)
            if data:
                self.procesar_datos(data['data'])
                return True
            print("💡 Intentando método alternativo...")
            return self.obtener_datos_alternativo()
        except Exception as e:
            print(f"❌ Error al obtener datos: {e}")
            print("💡 Intentando método alternativo...")
            return self.obtener_datos_alternativo()

    def obtener_datos_alternativo(self):
        print("\n🔄 Probando con método alternativo...")
        navegadores = ["chrome", "safari", "firefox", "edge"]
        for i, nav in enumerate(navegadores, 1):
            print(f"   Intento {i}/{len(navegadores)} (impersonate='{nav}')...")
            try:
                nueva_sesion = cffi_requests.Session(impersonate=nav)
                self.visitar_pagina_principal(nueva_sesion)
                time.sleep(random.uniform(2.0, 4.0))
                data = self._pedir_api(nueva_sesion)
                if data:
                    print(f"   ✅ Éxito con navegador impersonado: {nav}")
                    self.session = nueva_sesion
                    self.procesar_datos(data['data'])
                    return True
            except Exception as e:
                print(f"   ⚠️  Error: {str(e)[:80]}...")
        print("\n❌ No se pudo acceder a los datos después de varios intentos")
        print("💡 Sugerencias: cambia IP / desactiva VPN / pip install -U curl_cffi")
        return False

    def procesar_datos(self, datos):
        """Organiza sectores y acciones. La capitalización de cada sector se calcula
        sumando la de sus acciones (el campo 'value' del sector llega a 0)."""
        self.raw_data = datos
        self.acciones = []
        sectores_map = {}

        for item in datos:
            if 'parent' not in item:
                sectores_map[item['id']] = {
                    'nombre': item['name'],
                    'note': item.get('note', 0),
                    'value': 0.0,
                    'acciones': []
                }

        primera = True
        for item in datos:
            if 'parent' not in item:
                continue
            if primera:
                primera = False
            parent_id = item['parent']
            if parent_id not in sectores_map:
                continue
            ticker, nombre, link = self._extraer_campos(item)
            try:
                cap = float(item.get('value') or 0)
            except (TypeError, ValueError):
                cap = 0.0
            accion = {'ticker': ticker, 'nombre': nombre, 'valor': cap, 'link': link}
            sectores_map[parent_id]['acciones'].append(accion)
            sectores_map[parent_id]['value'] += cap
            self.acciones.append({
                'Sector': sectores_map[parent_id]['nombre'],
                'Ticker': ticker,
                'Nombre': nombre,
                'Capitalización': cap,
                'URL': link
            })

        self.sectores = sectores_map
        print(f"Se procesaron {len(self.sectores)} sectores y {len(self.acciones)} acciones")

    def volcar_json_crudo(self, archivo="ibex35_raw.json"):
        """Guarda el JSON tal como lo devuelve la API (útil para depurar cambios)."""
        with open(archivo, "w", encoding="utf-8") as f:
            json.dump(self.raw_data, f, indent=2, ensure_ascii=False)
        print(f"✅ JSON crudo guardado en: {archivo}")

    def mostrar_resumen(self):
        print("\n" + "=" * 80)
        print("RESUMEN DEL IBEX 35 POR SECTORES")
        print("=" * 80)
        for sector_data in self.sectores.values():
            print(f"\n📊 {sector_data['nombre']}")
            print(f"   Número de acciones: {len(sector_data['acciones'])}")
            print(f"   Capitalización total: {self._format_market_cap(sector_data['value'])}")
            print("   Acciones:")
            for a in sorted(sector_data['acciones'], key=lambda x: x['valor'], reverse=True):
                print(f"      • {a['ticker']:8} {a['nombre']:40} Cap: {self._format_market_cap(a['valor'])}")

    def guardar_csv(self, archivo='ibex35_heatmap.csv'):
        df = pd.DataFrame(self.acciones).sort_values(['Sector', 'Capitalización'], ascending=[True, False])
        df['Cap_Formateada'] = df['Capitalización'].apply(self._format_market_cap)
        df.to_csv(archivo, index=False, encoding='utf-8-sig')
        print(f"\n✅ Datos guardados en: {archivo}")

    def guardar_excel(self, archivo='ibex35_heatmap.xlsx'):
        with pd.ExcelWriter(archivo, engine='openpyxl') as writer:
            df_todas = pd.DataFrame(self.acciones).sort_values(['Sector', 'Capitalización'], ascending=[True, False])
            df_todas['Cap_Formateada'] = df_todas['Capitalización'].apply(self._format_market_cap)
            df_todas.to_excel(writer, sheet_name='Todas las acciones', index=False)

            resumen = [{
                'Sector': s['nombre'],
                'Número de Acciones': len(s['acciones']),
                'Capitalización Total': s['value'],
                'Cap_Formateada': self._format_market_cap(s['value'])
            } for s in self.sectores.values()]
            pd.DataFrame(resumen).sort_values('Capitalización Total', ascending=False) \
                .to_excel(writer, sheet_name='Resumen Sectores', index=False)

            for s in self.sectores.values():
                df_s = pd.DataFrame(s['acciones'])
                if df_s.empty:
                    continue
                df_s = df_s.rename(columns={'ticker': 'Ticker', 'nombre': 'Nombre',
                                            'valor': 'Capitalización', 'link': 'URL'}) \
                           .sort_values('Capitalización', ascending=False)
                df_s['Cap_Formateada'] = df_s['Capitalización'].apply(self._format_market_cap)
                df_s.to_excel(writer, sheet_name=s['nombre'][:31], index=False)
        print(f"✅ Datos guardados en Excel: {archivo}")

    def guardar_json(self, archivo='ibex35_heatmap.json'):
        resultado = {
            'fecha_extraccion': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'indice': 'IBEX 35',
            'total_acciones': len(self.acciones),
            'sectores': {s['nombre']: {
                'capitalizacion_total': s['value'],
                'cap_formateada': self._format_market_cap(s['value']),
                'numero_acciones': len(s['acciones']),
                'acciones': s['acciones']
            } for s in self.sectores.values()}
        }
        with open(archivo, 'w', encoding='utf-8') as f:
            json.dump(resultado, f, indent=2, ensure_ascii=False)
        print(f"✅ Datos guardados en JSON: {archivo}")

    def obtener_sector(self, nombre_sector):
        for s in self.sectores.values():
            if s['nombre'].lower() == nombre_sector.lower():
                return s['acciones']
        return None

    def buscar_accion(self, ticker_o_nombre):
        b = str(ticker_o_nombre).lower()
        return [a for a in self.acciones if b in a['Ticker'].lower() or b in a['Nombre'].lower()]

    def ejecutar(self, guardar_excel=True, guardar_csv=True, guardar_json=False, usar_ejemplo=False):
        print("=" * 80)
        print("EXTRACTOR DE DATOS IBEX 35 - MARKET SCREENER")
        print("=" * 80 + "\n")
        if not self.obtener_datos():
            print("\n❌ No se pudieron obtener los datos")
            return False
        self.mostrar_resumen()
        if guardar_excel:
            self.guardar_excel()
        if guardar_csv:
            self.guardar_csv()
        if guardar_json:
            self.guardar_json()
        print("\n" + "=" * 80)
        print("✅ PROCESO COMPLETADO EXITOSAMENTE")
        print("=" * 80)
        return True

    # ------------------------------------------------- YAHOO FINANCE: TICKERS Y BETAS
    def _mnemo_to_yahoo(self, mnemo):
        if not isinstance(mnemo, str) or not mnemo.strip():
            return None
        mn = mnemo.strip().upper()
        return self.yahoo_map_overrides.get(mn, f"{mn}.MC")

    def _get_info_yf(self, ticker, retries=3, sleep_sec=1.0):
        """Info completa de Yahoo (beta publicada, shortName...) con reintentos y caché."""
        if not ticker:
            return {}
        if ticker in self._cache_info:
            return self._cache_info[ticker]
        for intento in range(retries):
            try:
                info = yf.Ticker(ticker).get_info()
                if info:
                    self._cache_info[ticker] = info
                    return info
            except Exception:
                pass
            time.sleep(sleep_sec * (intento + 1))
        self._cache_info[ticker] = {}
        return {}

    def enriquecer_con_betas_y_cap(self):
        """
        - Añade YahooTicker y Beta_YF (beta publicada por Yahoo Finance)
        - Completa Nombre desde Yahoo si Market Screener no lo da
        - Ordena por sector y capitalización
        - Beta sectorial = media ponderada por capitalización
        """
        if not self.acciones:
            raise ValueError("No hay acciones cargadas. Ejecuta obtener_datos() primero.")

        df = pd.DataFrame(self.acciones).copy()
        df.rename(columns={"Capitalización": "Capitalizacion", "URL": "URL_MarketScreener"}, inplace=True)

        df["YahooTicker"] = df["Ticker"].apply(self._mnemo_to_yahoo)
        df["URL_Yahoo"] = df["YahooTicker"].apply(
            lambda t: f"https://finance.yahoo.com/quote/{t}" if t else "")

        print("\nDescargando datos de Yahoo Finance...")
        infos = {}
        for t in df["YahooTicker"]:
            if t and t not in infos:
                infos[t] = self._get_info_yf(t)
                time.sleep(0.35)  # evita rate-limit

        df["Beta_YF"] = pd.to_numeric(
            [infos.get(t, {}).get("beta") for t in df["YahooTicker"]], errors="coerce")
        df["Nombre"] = [
            n if n else (infos.get(t, {}).get("shortName") or "")
            for n, t in zip(df["Nombre"], df["YahooTicker"])
        ]

        df["Beta_YF"] = df["Beta_YF"].round(3)

        df = df.sort_values(["Sector", "Capitalizacion"], ascending=[True, False]).reset_index(drop=True)
        df["Cap_Formateada"] = df["Capitalizacion"].apply(self._format_market_cap)

        def _beta_pond(g):
            g = g[g["Beta_YF"].notna() & (g["Capitalizacion"] > 0)]
            if g.empty:
                return np.nan
            return (g["Capitalizacion"] * g["Beta_YF"]).sum() / g["Capitalizacion"].sum()

        filas = []
        for sector, g in df.groupby("Sector", dropna=False):
            cap_total = g["Capitalizacion"].sum()
            filas.append({
                "Sector": sector,
                "Capitalizacion_Total": cap_total,
                "Cap_Formateada": self._format_market_cap(cap_total),
                "N_Acciones": len(g),
                "N_con_Beta": int(g["Beta_YF"].notna().sum()),
                "Beta_Sector_Pond": round(_beta_pond(g), 3) if pd.notna(_beta_pond(g)) else np.nan,
            })
        sector_df = (pd.DataFrame(filas)
                     .sort_values("Capitalizacion_Total", ascending=False)
                     .reset_index(drop=True))

        self.df_empresas_enriq = df
        self.df_sectores_enriq = sector_df
        return df, sector_df
        print("="*80)

    # ------------------------------------------------------- PRESENTACIÓN Y EXPORTACIÓN
    COLS_EMPRESAS_PANTALLA = [ "Ticker", "YahooTicker", "Sector", "Nombre", "Cap_Formateada",
                              "Beta_YF"]
    COLS_SECTORES_PANTALLA = ["Sector", "Cap_Formateada", "N_Acciones", "N_con_Beta", "Beta_Sector_Pond"]

    def _empresas_para_excel(self, df):
        out = df.copy()
        out["Cap_MEUR"] = (out["Capitalizacion"] / 1_000_000).round(0)
        cols = ["Sector", "Ticker", "Nombre", "YahooTicker", "Cap_Formateada", "Cap_MEUR",
                "Beta_YF", "URL_Yahoo", "URL_MarketScreener"]
        return out[cols]

    def exportar_excel_enriquecido(self, archivo="ibex35_heatmap_enriquecido.xlsx"):
        if not hasattr(self, "df_empresas_enriq"):
            raise ValueError("Primero llama a enriquecer_con_betas_y_cap().")
        emp = self._empresas_para_excel(self.df_empresas_enriq)
        sec = self.df_sectores_enriq.copy()
        sec["Cap_MEUR"] = (sec["Capitalizacion_Total"] / 1_000_000).round(0)
        sec = sec[["Sector", "Cap_Formateada", "Cap_MEUR", "N_Acciones", "N_con_Beta", "Beta_Sector_Pond"]]
        with pd.ExcelWriter(archivo, engine="openpyxl") as writer:
            emp.to_excel(writer, sheet_name="Empresas (orden cap)", index=False)
            sec.to_excel(writer, sheet_name="Sectores (orden cap)", index=False)
            for sector in emp["Sector"].dropna().unique():
                emp[emp["Sector"] == sector].to_excel(writer, sheet_name=str(sector)[:31], index=False)
        print(f"✅ Excel enriquecido exportado: {archivo}")


# ==================== EJEMPLO DE USO ====================
if __name__ == "__main__":
    extractor = IBEXHeatmapExtractor()
    ok = extractor.ejecutar(guardar_excel=False, guardar_csv=False, guardar_json=False)

    if ok:
        # extractor.volcar_json_crudo()   # descomenta para depurar el JSON de la API
        df_emp, df_sec = extractor.enriquecer_con_betas_y_cap()

        print("\n=== SECTORES ORDENADOS POR CAPITALIZACIÓN (con Beta ponderada) ===")
        print("="*80)
        print(df_sec[extractor.COLS_SECTORES_PANTALLA].to_string(index=False))
        print("="*80)
        print("\n=== EMPRESAS POR SECTOR Y CAPITALIZACIÓN (con betas) ===")
        print("="*100)
        print(df_emp[extractor.COLS_EMPRESAS_PANTALLA].to_string(index=False))
        print("="*100)
        extractor.exportar_excel_enriquecido("IBEX35_Sectores_y_Betas.xlsx")

In [ ]:

# ==============================================================================
# CAPM IBEX 35 · UNIVERSO COMPLETO
# REGRESIONES OLS/HAC, COSTE DE CAPITAL Y ANÁLISIS
# ==============================================================================
from math import ceil
from pathlib import Path
from matplotlib.ticker import PercentFormatter

# ==============================================================================
# 01 · VALIDACIÓN Y CONFIGURACIÓN
# ==============================================================================
if "df_emp" not in globals():
    raise RuntimeError(
        "Ejecuta primero el extractor IBEX 35 para generar df_emp."
    )

MIN_OBS = 36
ANNUAL = globals().get("ANNUAL", 12)
N_PLOT = 10
NCOLS = 2

SCATTER_C = "steelblue"
LINE_C = "red"
GREY = "#9099A3"
INK = "#17263F"

OUT_DIR = Path(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

IND_DIR = OUT_DIR / "regresiones_individuales"
IND_DIR.mkdir(parents=True, exist_ok=True)


# ==============================================================================
# 02 · TASA LIBRE DE RIESGO Y MERCADO
# ==============================================================================
_rf_obs = rf_annual_month.dropna().sort_index()
if _rf_obs.empty:
    raise RuntimeError("No existen observaciones válidas de la rf BCE.")

RF_ANUAL = float(_rf_obs.iloc[-1])
RF_FECHA = _rf_obs.index[-1]

RF_ANUAL_ACTUAL = RF_ANUAL
RF_FECHA_ACTUAL = RF_FECHA

mkt_m = common[BENCHMARK]
rf_m = common["rf_m"]

mkt_ret_mean_ann = float(mkt_m.mean() * ANNUAL)
rf_mean_ann = float(rf_m.mean() * ANNUAL)
prima_hist_global = mkt_ret_mean_ann - rf_mean_ann

print("=" * 110)
print("CAPM IBEX 35 · PARÁMETROS DEL MERCADO")
print("=" * 110)
print(
    f"Periodo                  : "
    f"{common.index.min():%Y-%m} — {common.index.max():%Y-%m}"
)
print("Frecuencia               : Mensual")
print(f"Benchmark                : {BENCHMARK}")
print(f"Serie BCE                : {RF_PRESET}")
print(f"Última rf BCE            : {RF_ANUAL_ACTUAL:.2%}")
print(f"Fecha última rf          : {RF_FECHA_ACTUAL}")
print(f"Rf media histórica       : {rf_mean_ann:.2%}")
print(f"Retorno mercado anual    : {mkt_ret_mean_ann:.2%}")
print(f"Prima histórica IBEX TR  : {prima_hist_global:.2%}")
print("=" * 110)


# ==============================================================================
# 03 · UNIVERSO DINÁMICO IBEX 35
# ==============================================================================
uni = (
    df_emp.dropna(subset=["YahooTicker"])
    .sort_values("Capitalizacion", ascending=False)
    .drop_duplicates("YahooTicker")
)

UNIVERSE = {}
for r in uni.itertuples():
    nm = str(r.Nombre).strip() if pd.notna(r.Nombre) else ""
    UNIVERSE[r.YahooTicker] = nm if nm else r.YahooTicker

def nombre_empresa(tkr):
    nombre = " ".join(str(UNIVERSE[tkr]).split())

    if nombre.isupper():
        nombre = nombre.title()

    nombre = nombre.replace("S.a.", "S.A.")
    nombre = nombre.replace("S.l.", "S.L.")
    nombre = nombre.replace("Bbva", "BBVA")
    nombre = nombre.replace("Caixabank", "CaixaBank")

    return nombre

def ticker_corto(tkr):
    return str(tkr).split(".")[0].upper()

print(f"\nDescargando {len(UNIVERSE)} empresas...")
raw_u = yf.download(
    list(UNIVERSE),
    start=DOWNLOAD_START,
    end=END,
    auto_adjust=True,
    progress=False,
    threads=False,
    group_by="column"
)
if raw_u.empty:
    raise RuntimeError("Yahoo Finance no devolvió precios.")
if (
    isinstance(raw_u.columns, pd.MultiIndex)
    and "Close" in raw_u.columns.get_level_values(0)
):
    daily_u = raw_u["Close"].copy()
elif "Close" in raw_u.columns and len(UNIVERSE) == 1:
    daily_u = raw_u[["Close"]].copy()
    daily_u.columns = list(UNIVERSE)
else:
    raise RuntimeError("No se encuentra la columna Close.")

idx = pd.to_datetime(daily_u.index)

if idx.tz is not None:
    idx = idx.tz_localize(None)

daily_u.index = idx.normalize()

daily_u = (
    daily_u.sort_index()
    .apply(pd.to_numeric, errors="coerce")
)

prices_u = daily_u.resample("ME").last()

ultimo_mes_completo = (
    pd.Timestamp.today().to_period("M") - 1
).to_timestamp("M")

prices_u = prices_u.loc[
    prices_u.index <= ultimo_mes_completo
]
ret_u = prices_u.pct_change(
    fill_method=None
).reindex(common.index)

print(
    f"Meses con datos: "
    f"{ret_u.notna().any(axis=1).sum()}"
)

# ==============================================================================
# 04 · ESTIMACIÓN CAPM POR EMPRESA
# ==============================================================================
def fit_capm_ibex(tkr):
    """
    OLS con errores HAC:
        Ri - Rf = alpha + beta * (Rm - Rf) + epsilon

    Calcula:
        Ke histórico
        Ke usando la rf actual BCE
        Diferencia en puntos porcentuales
    """

    if tkr not in ret_u.columns:
        return None

    d = pd.concat(
        {
            "r": ret_u[tkr],
            "m": mkt_m,
            "rf": rf_m
        },
        axis=1
    ).dropna()

    if len(d) < MIN_OBS:
        return None

    x = (d["m"] - d["rf"]).astype(float)
    y = (d["r"] - d["rf"]).astype(float)

    if x.var() < 1e-12:
        return None

    fit = sm.OLS(
        y,
        sm.add_constant(x)
    ).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": 3}
    )
    alpha = float(fit.params.iloc[0])
    beta = float(fit.params.iloc[1])

    rf_a = float(d["rf"].mean() * ANNUAL)
    m_a = float(d["m"].mean() * ANNUAL)
    ret_a = float(d["r"].mean() * ANNUAL)

    vol = float(
        d["r"].std() * np.sqrt(ANNUAL)
    )

    prima_h = m_a - rf_a
    # CAPM histórico sobre la muestra individual
    ke_hist = rf_a + beta * prima_h
    # Escenario con rf actual:
    # beta y prima histórica permanecen constantes
    ke_rf_actual = RF_ANUAL_ACTUAL + beta * prima_h

    delta_ke_pp = 100 * (ke_rf_actual - ke_hist)

    row = {
        "Ticker": tkr,
        "Empresa": nombre_empresa(tkr),
        "N_meses": len(d),

        "Beta": beta,
        "Beta_cov": float(
            d["r"].cov(d["m"]) / d["m"].var()
        ),

        "Alfa_anual": alpha * ANNUAL,
        "P_alfa_HAC": float(fit.pvalues.iloc[0]),
        "P_beta_HAC": float(fit.pvalues.iloc[1]),
        "R2": float(fit.rsquared),

        "Volatilidad": vol,
        "Sharpe_Ratio": (
            (ret_a - rf_a) / vol
            if vol > 0 else np.nan
        ),

        "Retorno_anual": ret_a,

        "Rf_anual": rf_a,
        "Rf_media_hist": rf_a,
        "Prima_hist": prima_h,

        "Ke_hist": ke_hist,
        "Rf_actual": RF_ANUAL_ACTUAL,
        "Ke_rf_actual": ke_rf_actual,
        "Delta_Ke_pp": delta_ke_pp,
    }

    return row, x, y, fit

print("\n" + "=" * 110)
print("ESTIMACIÓN CAPM POR EMPRESA")
print("=" * 110)
rows = []
store = {}
excluded = []

for tkr in UNIVERSE:

    out = fit_capm_ibex(tkr)

    if out is None:
        d_ex = pd.concat(
            {
                "r": ret_u.get(
                    tkr,
                    pd.Series(
                        index=common.index,
                        dtype=float
                    )
                ),
                "m": mkt_m,
                "rf": rf_m
            },
            axis=1
        ).dropna()

        excluded.append({
            "Ticker": tkr,
            "Empresa": nombre_empresa(tkr),
            "Meses_validos": len(d_ex),
            "Motivo": (
                f"Menos de {MIN_OBS} meses válidos "
                "o datos insuficientes"
            )
        })
        continue

    row, x, y, fit = out

    rows.append(row)
    store[tkr] = (x, y, fit)

    print(
        f"{ticker_corto(tkr):<12} "
        f"{nombre_empresa(tkr):<34} "
        f"β={row['Beta']:>6.3f}  "
        f"R²={row['R2']:>6.3f}  "
        f"Ke hist.={row['Ke_hist']:>7.2%}  "
        f"Ke rf actual={row['Ke_rf_actual']:>7.2%}"
    )

if not rows:
    raise RuntimeError(
        "Ninguna regresión estimable. Revisa las descargas."
    )

excluded_df = pd.DataFrame(
    excluded,
    columns=[
        "Ticker",
        "Empresa",
        "Meses_validos",
        "Motivo"
    ]
)

res = pd.DataFrame(rows)[[
    "Ticker",
    "Empresa",
    "N_meses",
    "Beta",
    "Beta_cov",
    "Alfa_anual",
    "R2",
    "P_beta_HAC",
    "P_alfa_HAC",
    "Volatilidad",
    "Sharpe_Ratio",
    "Retorno_anual",
    "Rf_anual",
    "Rf_media_hist",
    "Prima_hist",
    "Ke_hist",
    "Rf_actual",
    "Ke_rf_actual",
    "Delta_Ke_pp"
]]

print("\n" + "=" * 110)
print(f"Empresas estimadas : {len(res)}")
print(f"Empresas excluidas : {len(excluded_df)}")
print("=" * 110)

if not excluded_df.empty:
    display(excluded_df)
# ==============================================================================
# 05 · FUNCIÓN ÚNICA DE REGRESIÓN SCL
# ==============================================================================

def draw_scl(
    ax,
    x,
    y,
    fit,
    compact=True,
    serie_label=None,
    ticker=None
):
    if serie_label is None:
        serie_label = ticker

    alpha = float(fit.params.iloc[0])
    beta = float(fit.params.iloc[1])

    grid = np.linspace(
        float(x.min()),
        float(x.max()),
        100
    )

    ax.scatter(
        x,
        y,
        s=16 if compact else 22,
        color=SCATTER_C,
        alpha=0.60,
        edgecolors="white",
        linewidths=0.25,
        label=(
            f"Datos {serie_label}"
            if serie_label else "Datos"
        )
    )

    ax.plot(
        grid,
        alpha + beta * grid,
        color=LINE_C,
        lw=2.3 if compact else 2.7,
        label=f"SCL: β={beta:.3f}"
    )

    ax.axhline(
        0,
        color="grey",
        lw=0.8,
        ls="--",
        alpha=0.7
    )

    ax.axvline(
        0,
        color="grey",
        lw=0.8,
        ls="--",
        alpha=0.7
    )

    ax.xaxis.set_major_formatter(
        PercentFormatter(1, decimals=0)
    )

    ax.yaxis.set_major_formatter(
        PercentFormatter(1, decimals=0)
    )

    ax.grid(
        True,
        alpha=0.25,
        ls=":"
    )

    ax.legend(
        loc="upper left",
        fontsize=9 if compact else 11
    )

# ==============================================================================
# 06 · PANEL SCL · 10 EMPRESAS
# ==============================================================================
print("\n" + "=" * 110)
print("PANEL DE REGRESIONES CAPM")
print("=" * 110)

n = min(N_PLOT, len(store))
nrows = ceil(n / NCOLS)

rng = np.random.default_rng()

tickers_plot = rng.choice(
    list(store.keys()),
    size=n,
    replace=False
)

fig, axs = plt.subplots(
    nrows,
    NCOLS,
    figsize=(14, 4.8 * nrows),
    constrained_layout=True,
    squeeze=False
)

axs = np.array(axs).reshape(-1)

for ax, tkr in zip(axs, tickers_plot):

    x, y, fit = store[tkr]
    label_emp = ticker_corto(tkr)

    alpha_anual = (
        float(fit.params.iloc[0]) * ANNUAL
    )

    beta = float(fit.params.iloc[1])
    r2 = float(fit.rsquared)

    draw_scl(
        ax,
        x,
        y,
        fit,
        compact=True,
        serie_label=label_emp
    )

    ax.set_title(
        f"Security Characteristic Line: {label_emp}\n"
        f"α={alpha_anual:.2%} (anual), "
        f"β={beta:.3f}, R²={r2:.3f}",
        fontsize=12,
        fontweight="bold",
        pad=13
    )

    ax.set_xlabel(
        f"Exceso mercado ({BENCHMARK})",
        fontsize=10
    )

    ax.set_ylabel(
        f"Exceso retorno {label_emp}",
        fontsize=10
    )

    ax.tick_params(labelsize=9)

for ax in axs[n:]:
    ax.axis("off")

fig.suptitle(
    "Regresiones CAPM | IBEX 35",
    fontsize=20,
    fontweight="bold",
    color=INK
)

panel_path = (
    OUT_DIR / "03_regresiones_capm_ibex35_panel.png"
)

fig.savefig(
    panel_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print(f"Panel guardado: {panel_path}")


# ==============================================================================
# 07 · REGRESIONES INDIVIDUALES
# ==============================================================================
print("\n" + "=" *110)
print("EXPORTACIÓN DE REGRESIONES INDIVIDUALES + EXPORTACIÓN DE RESULTADOS A EXCEL")
print("=" * 110)

for tkr, (x, y, fit) in store.items():

    label_emp = nombre_empresa(tkr)
    alpha_anual = (
        float(fit.params.iloc[0]) * ANNUAL
    )
    beta = float(fit.params.iloc[1])
    r2 = float(fit.rsquared)

    fig_ind, ax_ind = plt.subplots(
        figsize=(9, 6)
    )

    draw_scl(
        ax_ind,
        x,
        y,
        fit,
        compact=False,
        serie_label=ticker_corto(tkr)
    )

    ax_ind.set_title(
        f"Security Characteristic Line: {label_emp}\n"
        f"α={alpha_anual:.2%} (anual), "
        f"β={beta:.3f}, R²={r2:.3f}",
        fontsize=13,
        fontweight="bold",
        pad=15
    )

    ax_ind.set_xlabel(
        f"Exceso mercado ({BENCHMARK})",
        fontsize=12
    )

    ax_ind.set_ylabel(
        f"Exceso retorno {label_emp}",
        fontsize=12
    )

    ax_ind.tick_params(labelsize=10)
    fig_ind.tight_layout()

    filename = (
        f"SCL_{tkr.replace('.', '_')}.png"
    )

    fig_ind.savefig(
        IND_DIR / filename,
        dpi=180,
        bbox_inches="tight"
    )

    plt.close(fig_ind)

print(f"Regresiones individuales guardadas: {len(store)}")
print(f"Directorio: {IND_DIR}")

# ==============================================================================
# 08 · EXPORTACIÓN A EXCEL
# ==============================================================================
excel_path = OUT_DIR / "CAPM_IBEX35_Completo.xlsx"
info = pd.DataFrame({
    "Métrica": [
        "Rf anual última BCE",
        "Fecha última Rf",
        "Rf media histórica",
        "Retorno anual mercado",
        "Prima histórica mercado",
        "Rf actual escenario",
        "Metodología escenario",
        "Periodo inicio",
        "Periodo fin",
        "Meses",
        "Frecuencia",
        "Serie Rf"
    ],
    "Valor": [
        RF_ANUAL,
        str(RF_FECHA),
        rf_mean_ann,
        mkt_ret_mean_ann,
        prima_hist_global,
        RF_ANUAL_ACTUAL,
        "Rf actual + beta x prima histórica individual",
        f"{common.index.min():%Y-%m}",
        f"{common.index.max():%Y-%m}",
        len(common),
        "Mensual",
        RF_PRESET
    ]
})

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    res.to_excel(
        writer,
        sheet_name="Resumen_CAPM",
        index=False
    )

    info.to_excel(
        writer,
        sheet_name="Info_Mercado",
        index=False
    )

    ret_u.dropna(
        how="all",
        axis=1
    ).to_excel(
        writer,
        sheet_name="Retornos"
    )

    excluded_df.to_excel(
        writer,
        sheet_name="Excluidos",
        index=False
    )

print(f"Excel guardado: {excel_path}")
print("=" * 110)
# ==============================================================================
# 09 · ANÁLISIS DE RESULTADOS
# ==============================================================================
res_print = res.copy()
res_print["Empresa"] = (
    res_print["Empresa"]
    .astype(str)
    .str.split()
    .str.join(" ")
)

W = max(
    32,
    int(res_print["Empresa"].str.len().max())
)

SEP = "-" * (W + 82)


def encabezado(titulo):
    print("\n" + SEP)
    print(titulo)
    print(SEP)


encabezado("ANÁLISIS DE RESULTADOS · CAPM IBEX 35")
# ==============================================================================
# 09.1 · CLASIFICACIÓN POR BETA
# ==============================================================================
encabezado("CLASIFICACIÓN POR BETA")
print(
    f"{'Empresa':<{W}}  "
    f"{'Beta':>8}  "
    f"{'Clasificación':<16}"
)

print(SEP)

for _, row in res_print.sort_values(
    "Beta",
    ascending=False
).iterrows():

    beta = row["Beta"]

    tipo = (
        "AGRESIVA" if beta > 1
        else "DEFENSIVA"
    )

    print(
        f"{row['Empresa']:<{W}}  "
        f"{beta:>8.3f}  "
        f"{tipo:<16}"
    )

# ==============================================================================
# 09.2 · COSTE DE CAPITAL · COMPARATIVA RF
# ==============================================================================
encabezado("COSTE DE CAPITAL · RF HISTÓRICA VS ACTUAL BCE")
print(
    f"Rf actual BCE: {RF_ANUAL_ACTUAL:.2%} "
    f"({RF_FECHA_ACTUAL})"
)

print(
    f"Rf media global: {rf_mean_ann:.2%} | "
    f"Prima histórica IBEX TR: {prima_hist_global:.2%}"
)

print(SEP)
print(
    f"{'Empresa':<{W}}  "
    f"{'Beta':>7}  "
    f"{'Rf Hist.':>9}  "
    f"{'Rf Actual':>9}  "
    f"{'Ke Hist.':>10}  "
    f"{'Ke Actual':>10}  "
    f"{'Δ Ke (pp)':>10}"
)

print(SEP)
for _, row in res_print.sort_values(
    "Ke_hist",
    ascending=False
).iterrows():

    print(
        f"{row['Empresa']:<{W}}  "
        f"{row['Beta']:>7.3f}  "
        f"{row['Rf_media_hist']:>9.2%}  "
        f"{row['Rf_actual']:>9.2%}  "
        f"{row['Ke_hist']:>10.2%}  "
        f"{row['Ke_rf_actual']:>10.2%}  "
        f"{row['Delta_Ke_pp']:>+10.2f}"
    )

# ==============================================================================
# 09.3 · DETALLE KE CON RF ACTUAL
# ==============================================================================
encabezado("DETALLE DEL CÁLCULO · RF ACTUAL BCE")
print(
    f"{'Empresa':<{W}}  "
    f"{'Ke':>9}  "
    f"{'Rf Actual':>10}  "
    f"{'Beta':>8}  "
    f"{'Prima Hist.':>12}"
)

print(SEP)
for _, row in res_print.sort_values(
    "Ke_rf_actual",
    ascending=False
).iterrows():

    print(
        f"{row['Empresa']:<{W}}  "
        f"{row['Ke_rf_actual']:>9.2%}  "
        f"{row['Rf_actual']:>10.2%}  "
        f"{row['Beta']:>8.3f}  "
        f"{row['Prima_hist']:>12.2%}"
    )

print(
    "\nKe con rf actual = "
    "Rf actual BCE + Beta × Prima histórica individual"
)
print(
    "Escenario de sensibilidad; "
    "no constituye una predicción ex-ante independiente."
)

# ==============================================================================
# 09.4 · DETALLE KE HISTÓRICO
# ==============================================================================
encabezado("DETALLE DEL CÁLCULO · HISTÓRICO")
print(
    f"{'Empresa':<{W}}  "
    f"{'Ke':>9}  "
    f"{'Rf Media':>9}  "
    f"{'Beta':>8}  "
    f"{'Prima Hist.':>12}"
)

print(SEP)

for _, row in res_print.sort_values(
    "Ke_hist",
    ascending=False
).iterrows():

    print(
        f"{row['Empresa']:<{W}}  "
        f"{row['Ke_hist']:>9.2%}  "
        f"{row['Rf_media_hist']:>9.2%}  "
        f"{row['Beta']:>8.3f}  "
        f"{row['Prima_hist']:>12.2%}"
    )

print(
    "\nKe histórico = "
    "Rf media histórica + Beta × Prima histórica individual"
)

print(
    "Δ Ke (pp) = 100 × "
    "(Rf actual BCE − Rf media histórica individual)"
)


# ==============================================================================
# 09.5 · ALFAS ANUALIZADOS Y SIGNIFICACIÓN HAC
# ==============================================================================
encabezado("ALFAS ANUALIZADOS · P-VALOR HAC")
print(
    f"{'Empresa':<{W}}  "
    f"{'Alfa Anual':>13}  "
    f"{'P-valor HAC':>13}  "
    f"{'Significancia':>14}"
)

print(SEP)

for _, row in res_print.iterrows():

    p = row["P_alfa_HAC"]

    sig = (
        "***" if p < 0.01
        else "**" if p < 0.05
        else "*" if p < 0.10
        else "-"
    )

    print(
        f"{row['Empresa']:<{W}}  "
        f"{row['Alfa_anual']:>13.2%}  "
        f"{p:>13.4f}  "
        f"{sig:>14}"
    )

print(
    "\nSignificancia: "
    "*** p<0.01 | ** p<0.05 | * p<0.10"
)

# ==============================================================================
# 09.6 · CALIDAD DEL AJUSTE
# ==============================================================================
encabezado("CALIDAD DEL AJUSTE · R²")
print(
    f"{'Empresa':<{W}}  "
    f"{'R²':>8}  "
    f"{'Calidad':<12}"
)

print(SEP)
for _, row in res_print.sort_values(
    "R2",
    ascending=False
).iterrows():

    r2 = row["R2"]

    calidad = (
        "Excelente" if r2 > 0.7
        else "Buena" if r2 > 0.5
        else "Moderada" if r2 > 0.3
        else "Baja"
    )

    print(
        f"{row['Empresa']:<{W}}  "
        f"{r2:>8.3f}  "
        f"{calidad:<12}"
    )

# ==============================================================================
# 10 · TABLA COMPLETA DE RESULTADOS
# ==============================================================================
from IPython.display import display, HTML
encabezado("TABLA COMPLETA DE RESULTADOS")
# Ordenamos y preparamos la tabla
res_tabla = res.copy()

res_tabla["Empresa"] = (
    res_tabla["Empresa"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Formato numérico original
formatos = {
    "Beta": "{:.3f}",
    "Beta_cov": "{:.3f}",
    "Alfa_anual": "{:.2%}",
    "R2": "{:.3f}",
    "P_beta_HAC": "{:.4f}",
    "P_alfa_HAC": "{:.4f}",
    "Volatilidad": "{:.2%}",
    "Sharpe_Ratio": "{:.3f}",
    "Retorno_anual": "{:.2%}",
    "Rf_anual": "{:.2%}",
    "Rf_media_hist": "{:.2%}",
    "Prima_hist": "{:.2%}",
    "Ke_hist": "{:.2%}",
    "Rf_actual": "{:.2%}",
    "Ke_rf_actual": "{:.2%}",
    "Delta_Ke_pp": "{:+.2f}",
}

estilo = (
    res_tabla.style
    .format(formatos)
    .background_gradient(
        subset=["Beta"],
        cmap="RdYlGn_r"
    )
    .set_table_styles([
        {
            "selector": "table",
            "props": [
                ("width", "max-content"),
                ("min-width", "100%"),
                ("table-layout", "auto"),
                ("border-collapse", "collapse")
            ]
        },
        {
            "selector": "th, td",
            "props": [
                ("white-space", "nowrap"),
                ("word-break", "normal"),
                ("overflow-wrap", "normal"),
                ("padding", "7px 12px")
            ]
        },
        {
            "selector": "td.col1",
            "props": [
                ("min-width", "230px"),
                ("max-width", "none"),
                ("text-align", "left")
            ]
        },
    ])
)
print("="*610)
# Desplazamiento horizontal si la tabla supera el ancho disponible
display(HTML(
    '<div style="width:100%;overflow-x:auto;">'
    + estilo.to_html()
    + '</div>'
))
print("="*610)
# ==============================================================================
# 11 · RESUMEN FINAL
# ==============================================================================
print("\n" + SEP)
print("ANÁLISIS CAPM IBEX 35 COMPLETADO")
print(SEP)

print(f"Empresas estimadas         : {len(res)}")
print(f"Empresas excluidas         : {len(excluded_df)}")
print(f"Regresiones en el panel    : {n}")
print(f"Panel                      : {panel_path.name}")
print(f"Regresiones individuales   : {len(store)}")
print(f"Excel                      : {excel_path.name}")
print(f"Directorio                 : {OUT_DIR}")

print(SEP)

In [ ]:
# ==============================================================================
# SECURITY MARKET LINE (SML) - IBEX 35
# CAPM HISTÓRICO VS. RETORNO REALIZADO
#
# Universo: todas las empresas con CAPM estimado (34 actualmente)
#
# Estilo:
# - CAPM histórico: círculos azul acero
# - Retorno realizado: rombos dorados
# - Diferencias: líneas verticales grises
# - SML histórica global: línea turquesa
# - Etiquetas próximas a cada rombo, sin flechas
#
# Ejecutar después de la celda CAPM.
# Requiere: res, OUT_DIR, rf_mean_ann, prima_hist_global
# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

# ==============================================================================
# 01. CONFIGURACIÓN
# ==============================================================================

BLUE = "steelblue"
GOLD = "#D9A441"
GREY = "#89939E"
TEAL = "#009E91"
INK = "#17263F"

AJUSTAR_ETIQUETAS = True

OUT_DIR = Path(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Librería opcional para ajuste de etiquetas
try:
    from adjustText import adjust_text
    HAS_ADJUST_TEXT = True
except ImportError:
    HAS_ADJUST_TEXT = False

if AJUSTAR_ETIQUETAS and not HAS_ADJUST_TEXT:
    print(
        "adjustText no está instalado. "
        "Se utilizarán las etiquetas originales."
    )

# ==============================================================================
# 02. CONSTRUCCIÓN DEL RESUMEN
# ==============================================================================

summary_sml = res.copy()

summary_sml["CAPM_anual"] = summary_sml["Ke_hist"]
summary_sml["Real_anual_arit"] = summary_sml["Retorno_anual"]
summary_sml["Prima_anual"] = summary_sml["Prima_hist"]

summary_sml = summary_sml.dropna(
    subset=[
        "Ticker",
        "Beta",
        "CAPM_anual",
        "Real_anual_arit"
    ]
).copy()

summary_sml = (
    summary_sml
    .sort_values("Beta")
    .reset_index(drop=True)
)

summary_sml["Ticker_corto"] = (
    summary_sml["Ticker"]
    .astype(str)
    .str.split(".")
    .str[0]
    .str.upper()
)

n_empresas = len(summary_sml)

if n_empresas == 0:
    raise RuntimeError("No hay empresas válidas para representar la SML.")

print("=" * 90)
print("SECURITY MARKET LINE - IBEX 35")
print("=" * 90)
print(f"Empresas representadas : {n_empresas}")
print(f"Rf histórica global    : {rf_mean_ann:.2%}")
print(f"Prima histórica global : {prima_hist_global:.2%}")
print("=" * 90)

# ==============================================================================
# 03. PREPARACIÓN DE DATOS
# ==============================================================================

betas = summary_sml["Beta"].to_numpy(dtype=float)

capm_pct = (
    summary_sml["CAPM_anual"].to_numpy(dtype=float) * 100
)

real_pct = (
    summary_sml["Real_anual_arit"].to_numpy(dtype=float) * 100
)

tickers = summary_sml["Ticker_corto"].tolist()

# ==============================================================================
# 04. FIGURA
# ==============================================================================

fig, ax = plt.subplots(figsize=(16, 9))

# Diferencias verticales entre CAPM y rentabilidad realizada
ax.vlines(
    betas,
    ymin=np.minimum(capm_pct, real_pct),
    ymax=np.maximum(capm_pct, real_pct),
    colors=GREY,
    alpha=0.42,
    linewidth=1.3,
    zorder=2
)

# Puntos CAPM histórico
ax.scatter(
    betas,
    capm_pct,
    color=BLUE,
    s=75,
    zorder=4,
    label="CAPM histórico"
)

# Rombos retorno real
ax.scatter(
    betas,
    real_pct,
    color=GOLD,
    marker="D",
    s=65,
    zorder=5,
    label="Retorno real (aritmético)"
)

# ==============================================================================
# 05. SML HISTÓRICA GLOBAL
# ==============================================================================

x_min = min(0, float(betas.min()) - 0.12)
x_max = max(2.0, float(betas.max()) + 0.12)

xgrid = np.linspace(x_min, x_max, 250)

sml_global = 100 * (
    rf_mean_ann + xgrid * prima_hist_global
)

ax.plot(
    xgrid,
    sml_global,
    color=TEAL,
    lw=2.5,
    zorder=3,
    label="SML histórica global (referencia)"
)

# ==============================================================================
# 06. EJES, TÍTULO Y CUADRÍCULA
# ==============================================================================

ax.set_xlabel(
    "Beta estimada · riesgo sistemático",
    fontsize=12
)

ax.set_ylabel(
    "Retorno anual aritmético (%)",
    fontsize=12
)

ax.set_title(
    f"Security Market Line | IBEX 35 ({n_empresas} empresas)\n"
    "Retorno exigido por CAPM vs. retorno realizado",
    fontsize=17,
    fontweight="bold",
    pad=20,
    color=INK
)

ax.axhline(
    0,
    color=GREY,
    linewidth=0.8,
    linestyle="--",
    alpha=0.6
)

ax.axvline(
    0,
    color=GREY,
    linewidth=0.8,
    linestyle="--",
    alpha=0.6
)

# Localizadores de escala
ax.xaxis.set_major_locator(
    MaxNLocator(nbins=9)
)

ax.yaxis.set_major_locator(
    MaxNLocator(nbins=8)
)

ax.grid(
    True,
    alpha=0.22,
    linestyle=":"
)

ax.tick_params(labelsize=10)

# Márgenes fijos moderados
ax.set_xlim(x_min - 0.03, x_max + 0.06)

y_all = np.concatenate([
    capm_pct,
    real_pct,
    sml_global
])

y_min = float(y_all.min())
y_max = float(y_all.max())
y_range = max(y_max - y_min, 1.0)

ax.set_ylim(
    min(0, y_min - 0.06 * y_range),
    y_max + 0.10 * y_range
)

# ==============================================================================
# 07. ETIQUETAS DE LOS 34 TICKERS
# ==============================================================================

texts = []

# Etiquetas muy cerca de los rombos
for beta, real, ticker in zip(betas, real_pct, tickers):

    txt = ax.text(
        beta + 0.012,
        real + 0.35,
        ticker,
        fontsize=8.5,
        color=INK,
        alpha=0.92,
        ha="left",
        va="bottom",
        zorder=6
    )

    texts.append(txt)

# ==============================================================================
# 08. AJUSTE SUAVE DE ETIQUETAS
# ==============================================================================

# Ajustar únicamente en vertical.
# Sin flechas y sin desplazamientos horizontales.
#
# Se limita la fuerza del algoritmo para conservar
# el aspecto de la gráfica original.

if AJUSTAR_ETIQUETAS and HAS_ADJUST_TEXT:

    adjust_text(
        texts,
        ax=ax,

        # Movimiento de las etiquetas solo en Y
        only_move={
            "text": "y",
            "static": "y",
            "explode": "y",
            "pull": "y"
        },

        # Fuerzas suaves
        force_text=(0.0, 0.08),
        force_static=(0.0, 0.025),
        force_pull=(0.0, 0.03),
        force_explode=(0.0, 0.02),

        # Desplazamiento máximo por iteración
        max_move=(0, 2),

        # Ajuste limitado
        iter_lim=35,

        # No ampliar ejes automáticamente
        expand_axes=False,

        # Mantener dentro de los ejes
        ensure_inside_axes=True,

        # Sin arrowprops: no dibujar flechas
    )

# ==============================================================================
# 09. LEYENDA
# ==============================================================================

legend_handles = [

    Line2D(
        [], [],
        marker="o",
        linestyle="None",
        markerfacecolor=BLUE,
        markeredgecolor=BLUE,
        markersize=9,
        label="CAPM histórico"
    ),

    Line2D(
        [], [],
        marker="D",
        linestyle="None",
        markerfacecolor=GOLD,
        markeredgecolor=GOLD,
        markersize=8,
        label="Retorno real (aritmético)"
    ),

    Line2D(
        [], [],
        color=GREY,
        lw=1.5,
        label="Diferencia CAPM - real"
    ),

    Line2D(
        [], [],
        color=TEAL,
        lw=2.5,
        label="SML histórica global (referencia)"
    )
]

ax.legend(
    handles=legend_handles,
    loc="upper right",
    fontsize=11,
    frameon=True,
    facecolor="white",
    edgecolor="#DCE4EC"
)

# ==============================================================================
# 10. EXPORTACIÓN DEL GRÁFICO
# ==============================================================================

fig.tight_layout()

sml_path = OUT_DIR / "04_sml_historica_ibex35.png"

fig.savefig(
    sml_path,
    dpi=180,
    bbox_inches="tight"
)

plt.show()

print(f"\nGráfico SML guardado en: {sml_path}")

# ==============================================================================
# 11. TABLA COMPARATIVA
# ==============================================================================

summary_sml["Diferencia_real_CAPM"] = (
    summary_sml["Real_anual_arit"]
    - summary_sml["CAPM_anual"]
)

print("\n" + "=" * 110)
print("COMPARATIVA CAPM HISTÓRICO VS. RENTABILIDAD REAL")
print("=" * 110)

display(
    summary_sml[[
        "Ticker",
        "Empresa",
        "Beta",
        "N_meses",
        "CAPM_anual",
        "Real_anual_arit",
        "Diferencia_real_CAPM"
    ]].style.format({
        "Beta": "{:.3f}",
        "CAPM_anual": "{:.2%}",
        "Real_anual_arit": "{:.2%}",
        "Diferencia_real_CAPM": "{:+.2%}"
    }).background_gradient(
        subset=["Diferencia_real_CAPM"],
        cmap="RdYlGn"
    )
)

# ==============================================================================
# 12. RESUMEN FINAL
# ==============================================================================

print("\n" + "=" * 110)
print("ANÁLISIS SML COMPLETADO")
print("=" * 110)
print(f"Empresas representadas : {n_empresas}")
print(f"Rf histórica global    : {rf_mean_ann:.2%}")
print(f"Prima histórica global : {prima_hist_global:.2%}")
print(f"Gráfico guardado        : {sml_path}")
print("=" * 110)
